# Search Page Product Extractor

Extracts products from saved Coupang search-result page source: list price, discount rate and seller item IDs, then joins price-history data and builds a competitor-analysis workbook.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


> ⚠️ **Read before running.** This notebook sends automated requests to aiprice.com to pull price history for each product. That site's terms of service may prohibit automated access.
>
> It is shared to show the collection and parsing method, not as a tool to run against aiprice.com (a third-party price tracker). Do not run it against the live site without that site's permission. To follow the downstream steps, use the synthetic files in [`sample-data/`](../sample-data/).

# **쿠팡 경쟁사 분석 통합 코드**
검색결과 View-Source HTML → 상품 추출 → 상품 URL 확보/다운로드 → 정가·할인율 추출 → AiPrice 가격이력 → 제품분석 → 빈칸 보완 → 중복 제거
### **정가/할인율 기준**
- 매칭 기준: vendorItemId
- 정가: 검색결과 HTML priceArea.basePrice 우선, 없으면 i18nOriginalPrice.amount
- 할인율: 검색결과 HTML discountRate
- 기존 정가/할인율은 빈칸일 때만 입력
- 판매가격은 수정하지 않으며 마지막에 변경 여부를 검증
- 결과에 정가_크롤링상태를 기록
**### 실행**
아래 Code 셀을 위에서부터 순서대로 실행하면 됩니다.

In [ ]:
# ================================================================
# 0. 공통 환경 / 모든 의존성 선행 로드
# ================================================================

!pip -q install pandas openpyxl selenium beautifulsoup4 lxml

# ================================================================
# 기본 라이브러리
# ================================================================

import os
import re
import json
import math
import time
import shutil
import html as html_lib
import requests
import numpy as np
import pandas as pd

from datetime import datetime

from bs4 import BeautifulSoup

from urllib.parse import (
    urlparse,
    parse_qs,
    unquote,
    unquote_plus
)

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

# ================================================================
# Google Colab
# ================================================================

from google.colab import drive, files

# ================================================================
# Google Drive 연결
# ================================================================

print("=" * 80)
print("🔗 Google Drive 연결")
print("=" * 80)

drive.mount("/content/drive")

# ================================================================
# 상품 데이터 저장 경로
# ================================================================

DRIVE_PRODUCT_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/data_coupang"
)

os.makedirs(
    DRIVE_PRODUCT_DIR,
    exist_ok=True
)

print()
print("📁 Google Drive 상품 데이터 경로")
print(DRIVE_PRODUCT_DIR)

# ================================================================
# 로컬 상품 결과 저장 경로
# ================================================================

LOCAL_PRODUCT_DIR = "/content/coupang_product_results"

os.makedirs(
    LOCAL_PRODUCT_DIR,
    exist_ok=True
)

print()
print("📁 로컬 상품 결과 경로")
print(LOCAL_PRODUCT_DIR)

# ================================================================
# 쿠팡 검색결과 HTML 업로드
# ================================================================

print()
print("=" * 80)
print("📂 쿠팡 검색결과 HTML 파일을 모두 업로드하세요.")
print("=" * 80)

uploaded = files.upload()

# ================================================================
# 중요
# ------------------------------------------------
# 기존 ③ 셀이 사용하는 html_filenames 변수는
# 업로드된 파일명을 그대로 유지한다.
# ================================================================

html_filenames = list(uploaded.keys())

print()
print("=" * 80)
print("📄 업로드된 HTML 파일")
print("=" * 80)

print(
    f"업로드된 파일 수 : {len(html_filenames)}개"
)

for i, filename in enumerate(
    html_filenames,
    start=1
):
    print(
        f"{i:>3}. {filename}"
    )

# ================================================================
# 업로드 파일 확인
# ================================================================

if not html_filenames:
    raise RuntimeError(
        "❌ 업로드된 파일이 없습니다."
    )

print()
print("=" * 80)
print("✅ Cell 00 환경 설정 완료")
print("=" * 80)

In [ ]:
# ================================================================================
# ③ 실제 검색결과 Product 전체 추출 v7
# ================================================================================
# 핵심 변경
# - view-source 원문을 우선 분석
# - 복원본도 별도 분석
# - itemListElement 구조에만 의존하지 않음
# - vendorItemId / itemId 주변의 상품 JSON 직접 탐색
# - Product 타입이 없어도 상품 후보 복원
# - 검색결과 60개를 최대한 보존
# ================================================================================

import os
import re
import json
import math
import html as html_lib
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import (
    urlparse,
    parse_qs,
    unquote,
    unquote_plus
)

print("=" * 80)
print("③ 실제 검색결과 Product 전체 추출 v7")
print("=" * 80)


# ================================================================================
# 0. 업로드 파일 확인
# ================================================================================

if "html_filenames" not in globals():
    raise RuntimeError(
        "html_filenames 변수가 없습니다. Cell 00을 먼저 실행하세요."
    )

if not html_filenames:
    raise RuntimeError(
        "업로드된 HTML 파일이 없습니다."
    )

print()
print(
    f"📂 업로드된 HTML 파일 : "
    f"{len(html_filenames):,}개"
)

for i, filename in enumerate(
    html_filenames,
    1
):
    print(
        f"  {i}. {filename}"
    )


# ================================================================================
# 1. 파일명 검색어
# ================================================================================

def extract_keyword_from_filename(
    filename
):

    name = os.path.basename(
        filename
    )

    name = re.sub(
        r"\.html?$",
        "",
        name,
        flags=re.I
    )

    name = re.sub(
        r"^view-source_",
        "",
        name,
        flags=re.I
    )

    patterns = [
        r"(?:[?&])q=([^&]+)",
        r"(?:^|_)q=([^&]+)",
    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            name,
            flags=re.I
        )

        if m:

            keyword = unquote_plus(
                m.group(1)
            ).strip()

            if keyword:
                return keyword

    return None


# ================================================================================
# 2. HTML 검색어
# ================================================================================

def extract_keyword_from_html(
    source_html
):

    if not source_html:
        return None

    text = html_lib.unescape(
        source_html
    )

    # saved from url
    m = re.search(
        r"saved from url=\((?:\d+)\)(https?://[^\"<\s]+)",
        text,
        flags=re.I
    )

    if m:

        url = m.group(1)

        for _ in range(3):

            decoded = unquote(
                url
            )

            if decoded == url:
                break

            url = decoded

        try:

            parsed = urlparse(
                url
            )

            params = parse_qs(
                parsed.query
            )

            if "q" in params:

                keyword = unquote_plus(
                    params["q"][0]
                ).strip()

                if keyword:
                    return keyword

        except Exception:
            pass


    # 일반 URL
    matches = re.findall(
        r"https?://[^\"<\s]+[?&]q=([^&\"<\s]+)",
        text,
        flags=re.I
    )

    for value in matches:

        keyword = unquote_plus(
            value
        ).strip()

        if keyword:
            return keyword

    return None


def get_file_keyword(
    filename,
    source_html
):

    keyword = extract_keyword_from_html(
        source_html
    )

    if not keyword:

        keyword = extract_keyword_from_filename(
            filename
        )

    return (
        keyword.strip()
        if keyword
        else None
    )


# ================================================================================
# 3. view-source 복원
# ================================================================================

def restore_view_source(
    raw_text
):

    if not raw_text:
        return raw_text

    if "line-content" not in raw_text:
        return raw_text

    try:

        soup = BeautifulSoup(
            raw_text,
            "html.parser"
        )

        lines = soup.select(
            ".line-content"
        )

        if not lines:
            return raw_text

        restored = "".join(
            line.get_text(
                "",
                strip=False
            )
            for line in lines
        )

        if len(restored) > 1000:
            return restored

    except Exception as e:

        print(
            f"  ⚠️ view-source 복원 오류: {e}"
        )

    return raw_text


# ================================================================================
# 4. Source 정규화
# ================================================================================

def normalize_source_text(
    text
):

    if not text:
        return ""

    text = html_lib.unescape(
        text
    )

    # Chrome view-source에서 JSON이 escaped 된 경우
    text = text.replace(
        '\\"',
        '"'
    )

    text = text.replace(
        "\\u0026",
        "&"
    )

    text = text.replace(
        "\\u002F",
        "/"
    )

    text = text.replace(
        "\\u003D",
        "="
    )

    text = text.replace(
        "\\u003F",
        "?"
    )

    return text


# ================================================================================
# 5. Balanced JSON
# ================================================================================

def extract_balanced_json(
    text,
    start_pos
):

    if (
        start_pos < 0
        or start_pos >= len(text)
    ):
        return None

    opening = text[
        start_pos
    ]

    if opening not in "[{":
        return None

    closing = (
        "]"
        if opening == "["
        else "}"
    )

    depth = 0
    in_string = False
    escape = False

    for i in range(
        start_pos,
        len(text)
    ):

        ch = text[i]

        if in_string:

            if escape:

                escape = False

            elif ch == "\\":

                escape = True

            elif ch == '"':

                in_string = False

            continue

        if ch == '"':

            in_string = True
            continue

        if ch == opening:

            depth += 1

        elif ch == closing:

            depth -= 1

            if depth == 0:

                return text[
                    start_pos:i + 1
                ]

    return None


# ================================================================================
# 6. ID 추출
# ================================================================================

def clean_id(
    value
):

    if value is None:
        return None

    value = str(
        value
    ).strip()

    value = value.replace(
        "\\",
        ""
    )

    return (
        value
        if value
        else None
    )


def extract_id(
    text,
    key
):

    patterns = [

        rf'"{key}"\s*:\s*"(\d+)"',

        rf'"{key}"\s*:\s*(\d+)',

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            flags=re.I
        )

        if m:

            return clean_id(
                m.group(1)
            )

    return None


# ================================================================================
# 7. URL에서 ID
# ================================================================================

def get_id_from_url(
    url,
    key
):

    if not url:
        return None

    try:

        parsed = urlparse(
            str(url)
        )

        params = parse_qs(
            parsed.query
        )

        if key in params:

            return clean_id(
                params[key][0]
            )

    except Exception:
        pass

    return None


# ================================================================================
# 8. 상품 URL
# ================================================================================

def make_product_url(
    product
):

    url = product.get(
        "url"
    )

    if isinstance(
        url,
        dict
    ):

        url = (
            url.get("@id")
            or url.get("url")
        )

    if url:

        return str(
            url
        )

    return None


# ================================================================================
# 9. JSON-LD Product 추출
# ================================================================================

def extract_jsonld_products(
    text
):

    products = []


    # --------------------------------------------------------------------------
    # A. application/ld+json
    # --------------------------------------------------------------------------

    soup = BeautifulSoup(
        text,
        "html.parser"
    )

    scripts = soup.find_all(
        "script",
        attrs={
            "type":
                re.compile(
                    r"application/ld\+json",
                    re.I
                )
        }
    )


    for script in scripts:

        raw = script.string

        if not raw:

            raw = script.get_text(
                "",
                strip=False
            )

        if not raw:
            continue

        raw = html_lib.unescape(
            raw
        ).strip()

        try:

            data = json.loads(
                raw
            )

        except Exception:
            continue


        # ------------------------------------------------------
        # 재귀적으로 Product 찾기
        # ------------------------------------------------------

        def walk(
            obj
        ):

            if isinstance(
                obj,
                dict
            ):

                obj_type = obj.get(
                    "@type"
                )

                is_product = False

                if isinstance(
                    obj_type,
                    str
                ):

                    is_product = (
                        obj_type.lower()
                        == "product"
                    )

                elif isinstance(
                    obj_type,
                    list
                ):

                    is_product = any(
                        str(x).lower()
                        == "product"
                        for x in obj_type
                    )

                if is_product:

                    products.append(
                        obj
                    )

                for value in obj.values():

                    walk(
                        value
                    )

            elif isinstance(
                obj,
                list
            ):

                for value in obj:

                    walk(
                        value
                    )


        walk(
            data
        )


    return products


# ================================================================================
# 10. itemListElement Product
# ================================================================================

def extract_itemlist_products(
    text
):

    results = []


    matches = list(
        re.finditer(
            r'"itemListElement"\s*:',
            text,
            flags=re.I
        )
    )


    print(
        f"  🔍 itemListElement 후보 : "
        f"{len(matches):,}개"
    )


    for match in matches:

        array_start = text.find(
            "[",
            match.end(),
            min(
                len(text),
                match.end() + 20000
            )
        )

        if array_start == -1:
            continue


        raw_array = extract_balanced_json(
            text,
            array_start
        )

        if not raw_array:
            continue


        try:

            data = json.loads(
                raw_array
            )

        except Exception:
            continue


        if not isinstance(
            data,
            list
        ):
            continue


        current = []


        for item in data:

            if not isinstance(
                item,
                dict
            ):
                continue


            product = item.get(
                "item"
            )

            if not isinstance(
                product,
                dict
            ):
                continue


            # Product 타입이 없어도
            # name / url / ID가 있으면 후보로 인정
            has_product_type = (
                str(
                    product.get(
                        "@type",
                        ""
                    )
                ).lower()
                == "product"
            )

            has_name = bool(
                product.get(
                    "name"
                )
            )

            has_url = bool(
                product.get(
                    "url"
                )
            )

            if (
                has_product_type
                or has_name
                or has_url
            ):

                current.append(
                    {
                        "position":
                            item.get(
                                "position"
                            ),

                        "product":
                            product
                    }
                )


        if len(current) > len(
            results
        ):

            results = current


    print(
        f"  📦 itemList Product 후보 : "
        f"{len(results):,}개"
    )


    return results


# ================================================================================
# 11. vendorItemId 주변 상품 후보
# ================================================================================

def extract_vendor_candidates(
    text
):

    candidates = []


    # 모든 vendorItemId
    matches = list(
        re.finditer(
            r'"vendorItemId"\s*:\s*"(\d+)"',
            text,
            flags=re.I
        )
    )


    matches += list(
        re.finditer(
            r'"vendorItemId"\s*:\s*(\d+)',
            text,
            flags=re.I
        )
    )


    # 위치 중복 제거
    unique = {}

    for m in matches:

        unique[
            m.start()
        ] = m


    matches = sorted(
        unique.values(),
        key=lambda x: x.start()
    )


    print(
        f"  🔎 vendorItemId 후보 : "
        f"{len(matches):,}개"
    )


    for index, match in enumerate(
        matches
    ):

        vendor_id = match.group(
            1
        )

        pos = match.start()


        # 상품 하나가 보통 수천 자 이내에 존재
        left = max(
            0,
            pos - 12000
        )

        right = min(
            len(text),
            pos + 12000
        )

        block = text[
            left:right
        ]


        # ------------------------------------------------------
        # itemId
        # ------------------------------------------------------

        item_id = extract_id(
            block,
            "itemId"
        )


        # ------------------------------------------------------
        # productId
        # ------------------------------------------------------

        product_id = extract_id(
            block,
            "productId"
        )


        # ------------------------------------------------------
        # 상품 URL
        # ------------------------------------------------------

        url_match = re.search(
            r'https?://[^"\s<>]+(?:itemId|vendorItemId)=[^"\s<>]+',
            block,
            flags=re.I
        )

        product_url = (
            html_lib.unescape(
                url_match.group(0)
            )
            if url_match
            else None
        )


        # ------------------------------------------------------
        # URL에서 ID 보완
        # ------------------------------------------------------

        if not item_id:

            item_id = get_id_from_url(
                product_url,
                "itemId"
            )

        if not vendor_id:

            vendor_id = get_id_from_url(
                product_url,
                "vendorItemId"
            )


        if not product_id:

            product_id = get_id_from_url(
                product_url,
                "productId"
            )


        # ------------------------------------------------------
        # 상품명 후보
        # ------------------------------------------------------

        name = None


        name_patterns = [

            r'"name"\s*:\s*"([^"]{2,300})"',

            r'"productName"\s*:\s*"([^"]{2,300})"',

        ]


        for pattern in name_patterns:

            name_matches = list(
                re.finditer(
                    pattern,
                    block,
                    flags=re.I
                )
            )

            if name_matches:

                # vendorItemId와 가장 가까운 name
                nearest = min(
                    name_matches,
                    key=lambda x:
                        abs(
                            (
                                left
                                + x.start()
                            )
                            - pos
                        )
                )

                name = (
                    nearest
                    .group(1)
                    .strip()
                )

                break


        # ------------------------------------------------------
        # 가격 후보
        # ------------------------------------------------------

        price = None

        price_patterns = [

            r'"salePrice"\s*:\s*"([^"]+)"',

            r'"salePrice"\s*:\s*([0-9]+)',

            r'"price"\s*:\s*"([^"]+)"',

            r'"price"\s*:\s*([0-9]+)',

        ]


        for pattern in price_patterns:

            price_matches = list(
                re.finditer(
                    pattern,
                    block,
                    flags=re.I
                )
            )

            if price_matches:

                nearest = min(
                    price_matches,
                    key=lambda x:
                        abs(
                            (
                                left
                                + x.start()
                            )
                            - pos
                        )
                )

                price = (
                    nearest
                    .group(1)
                )

                break


        # ------------------------------------------------------
        # 리뷰
        # ------------------------------------------------------

        review_count = None

        review_patterns = [

            r'"reviewCount"\s*:\s*"([^"]+)"',

            r'"reviewCount"\s*:\s*([0-9]+)',

            r'"ratingCount"\s*:\s*"([^"]+)"',

            r'"ratingCount"\s*:\s*([0-9]+)',

        ]


        for pattern in review_patterns:

            review_matches = list(
                re.finditer(
                    pattern,
                    block,
                    flags=re.I
                )
            )

            if review_matches:

                nearest = min(
                    review_matches,
                    key=lambda x:
                        abs(
                            (
                                left
                                + x.start()
                            )
                            - pos
                        )
                )

                review_count = (
                    nearest
                    .group(1)
                )

                break


        # ------------------------------------------------------
        # 평점
        # ------------------------------------------------------

        rating = None

        rating_patterns = [

            r'"ratingValue"\s*:\s*"([^"]+)"',

            r'"ratingValue"\s*:\s*([0-9.]+)',

        ]


        for pattern in rating_patterns:

            rating_matches = list(
                re.finditer(
                    pattern,
                    block,
                    flags=re.I
                )
            )

            if rating_matches:

                nearest = min(
                    rating_matches,
                    key=lambda x:
                        abs(
                            (
                                left
                                + x.start()
                            )
                            - pos
                        )
                )

                rating = (
                    nearest
                    .group(1)
                )

                break


        # ------------------------------------------------------
        # 후보
        # ------------------------------------------------------

        candidates.append(
            {
                "검색순위":
                    index + 1,

                "상품명":
                    name,

                "가격":
                    price,

                "평점":
                    rating,

                "리뷰수":
                    review_count,

                "상품URL":
                    product_url,

                "productId":
                    product_id,

                "itemId":
                    item_id,

                "vendorItemId":
                    vendor_id,

                "이미지URL":
                    None,

            }
        )


    return candidates


# ================================================================================
# 12. JSON Product → row
# ================================================================================

def json_product_to_row(
    product,
    position,
    search_keyword
):

    url = make_product_url(
        product
    )

    product_id = (
        get_id_from_url(
            url,
            "productId"
        )
        or clean_id(
            product.get(
                "productId"
            )
        )
    )

    item_id = (
        get_id_from_url(
            url,
            "itemId"
        )
        or clean_id(
            product.get(
                "itemId"
            )
        )
    )

    vendor_id = (
        get_id_from_url(
            url,
            "vendorItemId"
        )
        or clean_id(
            product.get(
                "vendorItemId"
            )
        )
    )


    offers = product.get(
        "offers"
    )

    if isinstance(
        offers,
        list
    ):

        offers = (
            offers[0]
            if offers
            else {}
        )

    if not isinstance(
        offers,
        dict
    ):

        offers = {}


    rating_obj = product.get(
        "aggregateRating"
    )

    if not isinstance(
        rating_obj,
        dict
    ):

        rating_obj = {}


    image = product.get(
        "image"
    )

    if isinstance(
        image,
        list
    ):

        image = (
            image[0]
            if image
            else None
        )

    if isinstance(
        image,
        dict
    ):

        image = image.get(
            "url"
        )


    return {

        "검색키워드":
            search_keyword,

        "검색순위":
            position,

        "상품명":
            product.get(
                "name"
            ),

        "가격":
            (
                offers.get(
                    "price"
                )
                or product.get(
                    "price"
                )
            ),

        "평점":
            rating_obj.get(
                "ratingValue"
            ),

        "리뷰수":
            (
                rating_obj.get(
                    "reviewCount"
                )
                or rating_obj.get(
                    "ratingCount"
                )
            ),

        "상품URL":
            url,

        "productId":
            product_id,

        "itemId":
            item_id,

        "vendorItemId":
            vendor_id,

        "이미지URL":
            image,
    }


# ================================================================================
# 13. 숫자
# ================================================================================

def parse_money(
    value
):

    if value is None:
        return None

    text = str(
        value
    ).replace(
        ",",
        ""
    )

    m = re.search(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if not m:
        return None

    number = float(
        m.group(0)
    )

    return (
        int(number)
        if number.is_integer()
        else number
    )


# ================================================================================
# 14. 가격 영역
# ================================================================================

def extract_price_area(
    text,
    vendor_position
):

    left = max(
        0,
        vendor_position - 5000
    )

    right = min(
        len(text),
        vendor_position + 8000
    )

    area = text[
        left:right
    ]


    price_match = re.search(
        r'"priceArea"\s*:\s*\{',
        area,
        flags=re.I
    )

    if not price_match:

        # priceArea가 없어도 basePrice 등을 직접 탐색
        price_text = area

    else:

        price_text = area[
            price_match.start():
            price_match.start() + 6000
        ]


    # basePrice
    base_price = None

    for pattern in [

        r'"basePrice"\s*:\s*"([^"]*)"',
        r'"basePrice"\s*:\s*([0-9]+)',

    ]:

        m = re.search(
            pattern,
            price_text,
            flags=re.I
        )

        if m:

            base_price = parse_money(
                m.group(1)
            )

            break


    # Original
    original_price = None

    for pattern in [

        r'"i18nOriginalPrice"\s*:\s*\{.*?"amount"\s*:\s*"([^"]*)"',
        r'"i18nOriginalPrice"\s*:\s*\{.*?"amount"\s*:\s*([0-9]+)',

    ]:

        m = re.search(
            pattern,
            price_text,
            flags=re.I | re.DOTALL
        )

        if m:

            original_price = parse_money(
                m.group(1)
            )

            break


    # Sales
    sales_price = None

    for pattern in [

        r'"i18nSalesPrice"\s*:\s*\{.*?"amount"\s*:\s*"([^"]*)"',
        r'"i18nSalesPrice"\s*:\s*\{.*?"amount"\s*:\s*([0-9]+)',

    ]:

        m = re.search(
            pattern,
            price_text,
            flags=re.I | re.DOTALL
        )

        if m:

            sales_price = parse_money(
                m.group(1)
            )

            break


    # discountRate
    discount_rate = None

    m = re.search(
        r'"discountRate"\s*:\s*([0-9]+(?:\.[0-9]+)?)',
        price_text,
        flags=re.I
    )

    if m:

        discount_rate = float(
            m.group(1)
        )


    list_price = (
        base_price
        if base_price is not None
        else original_price
    )


    return {

        "정가":
            list_price,

        "basePrice":
            base_price,

        "i18nOriginalPrice":
            original_price,

        "판매가격_HTML":
            sales_price,

        "할인율":
            discount_rate,
    }


# ================================================================================
# 15. vendorItemId 가격 추출
# ================================================================================

def parse_html_vendor_prices(
    source_text,
    source_name
):

    text = normalize_source_text(
        source_text
    )

    results = {}


    matches = list(
        re.finditer(
            r'"vendorItemId"\s*:\s*"(\d+)"',
            text,
            flags=re.I
        )
    )

    matches += list(
        re.finditer(
            r'"vendorItemId"\s*:\s*(\d+)',
            text,
            flags=re.I
        )
    )


    unique = {}

    for m in matches:

        unique[
            m.start()
        ] = m


    matches = sorted(
        unique.values(),
        key=lambda x: x.start()
    )


    for m in matches:

        vendor_id = m.group(
            1
        )

        price_info = extract_price_area(
            text,
            m.start()
        )

        if price_info is None:
            continue


        item_id = None

        left = max(
            0,
            m.start() - 3000
        )

        right = min(
            len(text),
            m.start() + 3000
        )

        block = text[
            left:right
        ]


        item_id = extract_id(
            block,
            "itemId"
        )


        results[
            vendor_id
        ] = {

            "vendorItemId":
                vendor_id,

            "itemId":
                item_id,

            "정가":
                price_info[
                    "정가"
                ],

            "basePrice":
                price_info[
                    "basePrice"
                ],

            "i18nOriginalPrice":
                price_info[
                    "i18nOriginalPrice"
                ],

            "판매가격_HTML":
                price_info[
                    "판매가격_HTML"
                ],

            "할인율":
                price_info[
                    "할인율"
                ],

            "source":
                source_name,
        }


    return results


# ================================================================================
# 16. 전체 HTML 처리
# ================================================================================

all_product_rows = []

all_html_price_data = {}

file_results = []


print()
print("=" * 80)
print("📦 파일별 Product 추출")
print("=" * 80)


for file_idx, filename in enumerate(
    html_filenames,
    1
):

    print()
    print(
        f"[{file_idx}/{len(html_filenames)}] "
        f"{filename}"
    )


    # --------------------------------------------------------------------------
    # 원본 읽기
    # --------------------------------------------------------------------------

    with open(
        filename,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        raw_html = f.read()


    print(
        f"  HTML 원본 길이 : "
        f"{len(raw_html):,}자"
    )


    # --------------------------------------------------------------------------
    # 복원
    # --------------------------------------------------------------------------

    restored_html = restore_view_source(
        raw_html
    )


    print(
        f"  복원 HTML 길이 : "
        f"{len(restored_html):,}자"
    )


    # --------------------------------------------------------------------------
    # 검색어
    # --------------------------------------------------------------------------

    search_keyword = get_file_keyword(
        filename,
        raw_html
    )


    if not search_keyword:

        raise RuntimeError(
            f"검색어 추출 실패: {filename}"
        )


    print(
        f"  🔎 검색어 : "
        f"{search_keyword}"
    )


    # ==========================================================================
    # A. 원본 / 복원본 모두 분석
    # ==========================================================================

    source_candidates = [
        (
            "원본",
            raw_html
        ),
        (
            "복원",
            restored_html
        )
    ]


    selected_rows = []


    for source_label, source_text in (
        source_candidates
    ):

        normalized = normalize_source_text(
            source_text
        )


        print()
        print(
            f"  [{source_label}] "
            f"상품 구조 분석"
        )


        # ----------------------------------------------------------------------
        # JSON-LD Product
        # ----------------------------------------------------------------------

        json_products = (
            extract_jsonld_products(
                source_text
            )
        )


        print(
            f"    JSON-LD Product : "
            f"{len(json_products):,}"
        )


        if len(json_products) > len(
            selected_rows
        ):

            json_rows = []

            for idx, product in enumerate(
                json_products,
                1
            ):

                json_rows.append(
                    json_product_to_row(
                        product,
                        idx,
                        search_keyword
                    )
                )

            selected_rows = json_rows


        # ----------------------------------------------------------------------
        # itemListElement
        # ----------------------------------------------------------------------

        itemlist = (
            extract_itemlist_products(
                normalized
            )
        )


        if len(itemlist) > len(
            selected_rows
        ):

            item_rows = []

            for idx, item in enumerate(
                itemlist,
                1
            ):

                position = item.get(
                    "position"
                )

                try:

                    position = int(
                        position
                    )

                except Exception:

                    position = idx


                item_rows.append(
                    json_product_to_row(
                        item["product"],
                        position,
                        search_keyword
                    )
                )


            selected_rows = item_rows


        # ----------------------------------------------------------------------
        # vendorItemId fallback
        # ----------------------------------------------------------------------

        vendor_rows = (
            extract_vendor_candidates(
                normalized
            )
        )


        print(
            f"    vendorItem 후보 : "
            f"{len(vendor_rows):,}"
        )


        if len(vendor_rows) > len(
            selected_rows
        ):

            for row in vendor_rows:

                row[
                    "검색키워드"
                ] = search_keyword

            selected_rows = vendor_rows


    # ==========================================================================
    # B. 결과 정리
    # ==========================================================================

    # vendorItemId가 있는 행 우선
    if selected_rows:

        valid_rows = []

        for row in selected_rows:

            if row.get(
                "vendorItemId"
            ):

                valid_rows.append(
                    row
                )


        # vendorItemId가 있는 결과가 있으면 그것 사용
        if valid_rows:

            selected_rows = valid_rows


    # --------------------------------------------------------------------------
    # 순위 정리
    # --------------------------------------------------------------------------

    for idx, row in enumerate(
        selected_rows,
        1
    ):

        try:

            rank = int(
                row.get(
                    "검색순위",
                    idx
                )
            )

        except Exception:

            rank = idx


        row[
            "검색순위"
        ] = rank

        row[
            "검색키워드"
        ] = search_keyword


    # --------------------------------------------------------------------------
    # 중복 vendorItemId 제거
    # --------------------------------------------------------------------------

    dedup = {}

    for row in selected_rows:

        vendor_id = row.get(
            "vendorItemId"
        )

        if not vendor_id:

            continue

        if vendor_id not in dedup:

            dedup[
                vendor_id
            ] = row

        else:

            old = dedup[
                vendor_id
            ]

            # 정보가 더 많은 행 선택
            old_score = sum(
                bool(
                    old.get(
                        col
                    )
                )
                for col in [
                    "상품명",
                    "상품URL",
                    "itemId",
                    "productId",
                    "가격",
                    "리뷰수",
                ]
            )

            new_score = sum(
                bool(
                    row.get(
                        col
                    )
                )
                for col in [
                    "상품명",
                    "상품URL",
                    "itemId",
                    "productId",
                    "가격",
                    "리뷰수",
                ]
            )

            if new_score > old_score:

                dedup[
                    vendor_id
                ] = row


    selected_rows = list(
        dedup.values()
    )


    # --------------------------------------------------------------------------
    # 순위 정렬
    # --------------------------------------------------------------------------

    if selected_rows:

        for idx, row in enumerate(
            selected_rows,
            1
        ):

            try:

                row[
                    "검색순위"
                ] = int(
                    row.get(
                        "검색순위",
                        idx
                    )
                )

            except Exception:

                row[
                    "검색순위"
                ] = idx


        selected_rows = sorted(
            selected_rows,
            key=lambda x:
                x.get(
                    "검색순위",
                    999999
                )
        )


    # --------------------------------------------------------------------------
    # 전체 결과
    # --------------------------------------------------------------------------

    all_product_rows.extend(
        selected_rows
    )


    # --------------------------------------------------------------------------
    # 가격 데이터
    # --------------------------------------------------------------------------

    price_data = parse_html_vendor_prices(
        restored_html,
        filename
    )


    for vendor_id, record in (
        price_data.items()
    ):

        if vendor_id not in all_html_price_data:

            all_html_price_data[
                vendor_id
            ] = record


    print()
    print(
        f"  ✅ 최종 Product : "
        f"{len(selected_rows):,}개"
    )

    print(
        f"  💰 정가/할인율 vendorItemId : "
        f"{len(price_data):,}개"
    )


    file_results.append(
        {
            "파일명":
                filename,

            "검색키워드":
                search_keyword,

            "상품수":
                len(selected_rows),

            "HTML길이":
                len(raw_html),

            "복원HTML길이":
                len(restored_html),

            "정가추출수":
                len(price_data),
        }
    )


# ================================================================================
# 17. DataFrame
# ================================================================================

jsonld_products_df = pd.DataFrame(
    all_product_rows
)


# ================================================================================
# 18. 컬럼 순서
# ================================================================================

base_columns = [

    "검색키워드",
    "검색순위",
    "상품명",
    "가격",
    "평점",
    "리뷰수",
    "상품URL",
    "productId",
    "itemId",
    "vendorItemId",
    "이미지URL",

]


if not jsonld_products_df.empty:

    for col in base_columns:

        if col not in jsonld_products_df.columns:

            jsonld_products_df[
                col
            ] = None


    jsonld_products_df = (
        jsonld_products_df[
            base_columns
        ]
    )


    # 숫자
    jsonld_products_df[
        "검색순위"
    ] = pd.to_numeric(
        jsonld_products_df[
            "검색순위"
        ],
        errors="coerce"
    )


    jsonld_products_df = (
        jsonld_products_df
        .sort_values(
            [
                "검색키워드",
                "검색순위"
            ],
            na_position="last"
        )
        .reset_index(
            drop=True
        )
    )


# ================================================================================
# 19. AiPrice 대상
# ================================================================================

if not jsonld_products_df.empty:

    ai_price_products_df = (
        jsonld_products_df[
            jsonld_products_df[
                "vendorItemId"
            ].notna()
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

else:

    ai_price_products_df = pd.DataFrame(
        columns=base_columns
    )


# ================================================================================
# 20. 파일별 결과
# ================================================================================

file_result_df = pd.DataFrame(
    file_results
)


print()
print("=" * 80)
print("📊 파일별 결과")
print("=" * 80)

display(
    file_result_df
)


# ================================================================================
# 21. 검색어별 결과
# ================================================================================

print()
print("=" * 80)
print("🔎 검색어별 Product 수")
print("=" * 80)

if not jsonld_products_df.empty:

    print(
        jsonld_products_df[
            "검색키워드"
        ]
        .value_counts()
        .to_string()
    )


# ================================================================================
# 22. 최종 결과
# ================================================================================

print()
print("=" * 80)
print("③ 전체 Product 추출 결과")
print("=" * 80)

print(
    f"업로드 HTML 파일 : "
    f"{len(html_filenames):,}개"
)

print(
    f"전체 Product      : "
    f"{len(jsonld_products_df):,}개"
)

print(
    f"vendorItemId      : "
    f"{len(ai_price_products_df):,}개"
)


# ================================================================================
# 23. 샘플
# ================================================================================

if not ai_price_products_df.empty:

    print()
    print("📋 Product 샘플")

    display(
        ai_price_products_df[
            [
                "검색키워드",
                "검색순위",
                "상품명",
                "vendorItemId",
                "itemId",
                "상품URL",
                "가격",
                "리뷰수",
            ]
        ].head(20)
    )


# ================================================================================
# 24. 최종 검증
# ================================================================================

expected_count = (
    len(html_filenames)
    * 60
)


print()
print("=" * 80)
print("✅ 최종 검증")
print("=" * 80)

print(
    f"예상 Product 수 : "
    f"{expected_count:,}"
)

print(
    f"실제 Product 수 : "
    f"{len(jsonld_products_df):,}"
)

print(
    f"AiPrice 대상    : "
    f"{len(ai_price_products_df):,}"
)


if len(
    jsonld_products_df
) == expected_count:

    print()
    print(
        "🎉 정상 완료!"
    )

elif len(
    jsonld_products_df
) > 0:

    print()
    print(
        "⚠️ Product는 추출됐지만 "
        "예상 개수와 다릅니다."
    )

else:

    print()
    print(
        "❌ Product가 0개입니다."
    )

    print(
        "이번에는 원본 HTML의 "
        "vendorItemId 구조까지 확인해야 합니다."
    )


# ================================================================================
# 25. 다음 셀에서 사용할 변수
# ================================================================================

print()
print("=" * 80)
print("📌 다음 셀 사용 변수")
print("=" * 80)

print("jsonld_products_df")
print("ai_price_products_df")
print("all_html_price_data")
print("file_result_df")

print()
print("③ 완료")

In [ ]:
# 20. AiPrice 대상
# ================================================================================

ai_price_products_df = (
    jsonld_products_df[
        jsonld_products_df[
            "vendorItemId"
        ].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ================================================================================
# 21. AiPrice 대상 결과
# ================================================================================

print()
print("=" * 80)
print("iv AiPrice 대상 생성 완료")
print("=" * 80)

print(
    f"전체 검색결과       : "
    f"{len(jsonld_products_df):,}"
)

print(
    f"vendorItemId 확보   : "
    f"{len(ai_price_products_df):,}"
)


# ================================================================================
# 22. AiPrice 대상 검색어별 확인
# ================================================================================

if not ai_price_products_df.empty:

    print()
    print(
        "검색키워드별 AiPrice 대상:"
    )

    print(
        ai_price_products_df[
            "검색키워드"
        ]
        .value_counts()
        .to_string()
    )


# ================================================================================
# 23. 샘플
# ================================================================================

if not ai_price_products_df.empty:

    print()
    print("AiPrice 대상 샘플:")

    display(
        ai_price_products_df[
            [
                "검색키워드",
                "검색순위",
                "상품명",
                "productId",
                "itemId",
                "vendorItemId"
            ]
        ]
        .head(20)
    )


# ================================================================================
# 24. 최종 검증
# ================================================================================

expected_count = (
    len(html_filenames) * 60
)

print()
print("=" * 80)
print("✅ 최종 검증")
print("=" * 80)

print(
    f"예상 Product 수      : "
    f"{expected_count:,}"
)

print(
    f"실제 Product 수      : "
    f"{len(jsonld_products_df):,}"
)

print(
    f"실제 AiPrice 대상    : "
    f"{len(ai_price_products_df):,}"
)


if len(jsonld_products_df) == expected_count:

    print(
        "\n🎉 정상: "
        f"{len(html_filenames)}개 HTML × 60개 = "
        f"{expected_count}개 Product 추출 완료"
    )

elif len(jsonld_products_df) > 0:

    print(
        "\n⚠️ Product는 추출됐지만 "
        "파일당 60개 기준과 다릅니다."
    )

else:

    raise RuntimeError(
        "❌ Product가 하나도 추출되지 않았습니다."
    )

# ================================================================================


In [ ]:
# iv. AiPrice 180일 가격 이력 전체 수집
# ================================================================================

import requests
import pandas as pd
import json
import time
import re
from datetime import datetime

print("=" * 80)
print("iv AiPrice 180일 가격 이력 전체 수집")
print("=" * 80)

# ------------------------------------------------------------------------------
# 1. AiPrice 설정
# ------------------------------------------------------------------------------

AIPRICE_URL = "https://www.aiprice.com/Search/priceTracking.html"

AIPRICE_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.aiprice.com/",
    "X-Requested-With": "XMLHttpRequest",
    "Connection": "keep-alive",
}

session = requests.Session()
session.headers.update(AIPRICE_HEADERS)


# ------------------------------------------------------------------------------
# 2. timestamp → datetime 변환
# ------------------------------------------------------------------------------

def parse_aiprice_time(value):

    if value is None:
        return pd.NaT

    try:
        # 숫자로 변환
        if isinstance(value, (int, float)):
            ts = float(value)

            # milliseconds
            if ts > 1e12:
                return pd.to_datetime(ts, unit="ms", errors="coerce")

            # seconds
            elif ts > 1e9:
                return pd.to_datetime(ts, unit="s", errors="coerce")

            else:
                return pd.to_datetime(ts, unit="s", errors="coerce")

        # 문자열 timestamp
        s = str(value).strip()

        if s.isdigit():
            ts = float(s)

            if ts > 1e12:
                return pd.to_datetime(ts, unit="ms", errors="coerce")

            elif ts > 1e9:
                return pd.to_datetime(ts, unit="s", errors="coerce")

        return pd.to_datetime(s, errors="coerce")

    except Exception:
        return pd.NaT


# ------------------------------------------------------------------------------
# 3. 가격 문자열 변환
# ------------------------------------------------------------------------------

def parse_price(value):

    if value is None:
        return None

    try:
        if isinstance(value, (int, float)):
            return float(value)

        s = str(value)
        s = re.sub(r"[^0-9.\-]", "", s)

        if not s:
            return None

        return float(s)

    except Exception:
        return None


# ------------------------------------------------------------------------------
# 4. AiPrice 응답에서 가격 이력 추출
# ------------------------------------------------------------------------------

def extract_price_history(data):

    records = []

    def walk(obj):

        if isinstance(obj, dict):

            # 가격 이력 한 건
            if "time_update" in obj and "price" in obj:

                records.append({
                    "가격": parse_price(obj.get("price")),
                    "m_price": parse_price(obj.get("m_price")),
                    "time_update": obj.get("time_update"),
                    "최저가": parse_price(obj.get("lowprice")),
                    "adid": obj.get("adid"),
                    "price_symbol": obj.get("price_symbol"),
                })

            for value in obj.values():
                walk(value)

        elif isinstance(obj, list):

            for value in obj:
                walk(value)

    walk(data)

    return records


# ------------------------------------------------------------------------------
# 5. vendorItemId 중복 제거
# ------------------------------------------------------------------------------

work_df = ai_price_products_df.copy()

print("\n" + "=" * 80)
print("iv 시작 직전 ai_price_products_df 검수")
print("=" * 80)

print("ai_price_products_df 행 수 :", len(ai_price_products_df))
print("ai_price_products_df 컬럼 :", ai_price_products_df.columns.tolist())

if "검색순위" in ai_price_products_df.columns:
    print(
        "검색순위 :",
        ai_price_products_df["검색순위"].tolist()
    )

if "vendorItemId" in ai_price_products_df.columns:
    print(
        "vendorItemId 개수 :",
        ai_price_products_df["vendorItemId"].nunique()
    )

print("\n상품 샘플:")
print(
    ai_price_products_df[
        ["검색순위", "상품명", "vendorItemId"]
    ].head(20).to_string(index=False)
)

work_df["vendorItemId"] = (
    work_df["vendorItemId"]
    .astype(str)
    .str.strip()
)

before_count = len(work_df)

work_unique_df = (
    work_df[
        work_df["vendorItemId"].notna()
        & (work_df["vendorItemId"] != "")
        & (work_df["vendorItemId"] != "nan")
    ]
    .drop_duplicates(subset=["vendorItemId"], keep="first")
    .copy()
)

after_count = len(work_unique_df)

print(f"검색결과 상품 수       : {before_count}")
print(f"중복 제거 후 상품 수   : {after_count}")
print(f"중복 요청 방지 수      : {before_count - after_count}")
print()


# ------------------------------------------------------------------------------
# 6. 전체 AiPrice 수집
# ------------------------------------------------------------------------------

all_history = []

success_count = 0
fail_count = 0

for idx, row in work_unique_df.reset_index(drop=True).iterrows():

    product_name = str(row.get("상품명", ""))

    vendor_item_id = str(row["vendorItemId"]).strip()

    params = {
        "sku_id": vendor_item_id,
        "adid": "206",
        "currency": "KRW",
        "day": "180",
    }

    try:

        response = session.post(
            AIPRICE_URL,
            params=params,
            timeout=30
        )

        if response.status_code != 200:

            fail_count += 1

            print(
                f"[{idx+1}/{len(work_unique_df)}] "
                f"실패 | HTTP {response.status_code} | "
                f"vendorItemId={vendor_item_id}"
            )

            time.sleep(0.7)
            continue

        # JSON 변환
        try:
            data = response.json()

        except Exception:

            data = json.loads(response.text)

        history = extract_price_history(data)

        if not history:

            fail_count += 1

            print(
                f"[{idx+1}/{len(work_unique_df)}] "
                f"이력 없음 | vendorItemId={vendor_item_id}"
            )

            time.sleep(0.7)
            continue

        # 상품 기본정보 + 가격 이력 결합
        for h in history:

            all_history.append({
                "검색순위": row.get("검색순위"),
                "검색키워드": row.get("검색키워드"),
                "상품명": row.get("상품명"),
                "상품URL": row.get("상품URL"),

                "productId": row.get("productId"),
                "itemId": row.get("itemId"),
                "vendorItemId": vendor_item_id,

                "가격": h.get("가격"),
                "m_price": h.get("m_price"),
                "time_update": h.get("time_update"),
                "최저가": h.get("최저가"),
                "adid": h.get("adid"),
                "price_symbol": h.get("price_symbol"),
            })

        success_count += 1

        print(
            f"[{idx+1}/{len(work_unique_df)}] "
            f"성공 | 이력 {len(history)}건 | "
            f"{product_name[:45]}"
        )

    except Exception as e:

        fail_count += 1

        print(
            f"[{idx+1}/{len(work_unique_df)}] "
            f"오류 | {type(e).__name__}: {e}"
        )

    # AiPrice 서버 부담 방지
    time.sleep(0.7)


# ------------------------------------------------------------------------------
# 7. DataFrame 생성
# ------------------------------------------------------------------------------

ai_price_history_df = pd.DataFrame(all_history)

print()
print("=" * 80)
print("iv AiPrice 수집 결과")
print("=" * 80)

print(f"요청 상품 수       : {len(work_unique_df)}")
print(f"성공 상품 수       : {success_count}")
print(f"실패 상품 수       : {fail_count}")
print(f"전체 가격 이력 수  : {len(ai_price_history_df):,}")


# ------------------------------------------------------------------------------
# 8. 날짜 변환
# ------------------------------------------------------------------------------

if not ai_price_history_df.empty:

    ai_price_history_df["날짜"] = ai_price_history_df["time_update"].apply(
        parse_aiprice_time
    )

    ai_price_history_df["날짜"] = pd.to_datetime(
        ai_price_history_df["날짜"],
        errors="coerce"
    )

    ai_price_history_df = ai_price_history_df.sort_values(
        ["검색순위", "vendorItemId", "날짜"]
    ).reset_index(drop=True)


# ------------------------------------------------------------------------------
# 9. 결과 확인
# ------------------------------------------------------------------------------

print()
print("샘플 데이터")
print("-" * 80)

if not ai_price_history_df.empty:

    display(
        ai_price_history_df[
            [
                "검색순위",
                "상품명",
                "vendorItemId",
                "날짜",
                "가격",
                "최저가",
                "m_price",
            ]
        ].head(20)
    )

else:

    print("가격 이력이 없습니다.")

# ================================================================================
# iv-A 상품별 AiPrice 180일 가격 요약
# ================================================================================

print("=" * 80)
print("iv-A 상품별 AiPrice 180일 가격 요약")
print("=" * 80)

if ai_price_history_df.empty:
    ai_price_summary_df = pd.DataFrame()
    print("가격 이력이 없습니다.")

else:

    # --------------------------------------------------------------------------
    # 0. iv-A 시작 데이터 검수
    # --------------------------------------------------------------------------

    print("\n[iv-A 시작 데이터 검수]")
    print("ai_price_history_df 행 수 :", len(ai_price_history_df))
    print("ai_price_history_df 컬럼 :")
    print(ai_price_history_df.columns.tolist())

    if "vendorItemId" in ai_price_history_df.columns:
        print(
            "vendorItemId 고유 개수 :",
            ai_price_history_df["vendorItemId"].nunique()
        )

    if "검색순위" in ai_price_history_df.columns:
        print(
            "검색순위 개수 :",
            ai_price_history_df["검색순위"].nunique()
        )

    print("\n상위 10개:")
    print(
        ai_price_history_df.head(10).to_string(index=False)
    )


    # --------------------------------------------------------------------------
    # 1. 현재 쿠팡 가격 컬럼 확인
    # --------------------------------------------------------------------------

    print("현재 상품 데이터 컬럼:")
    print(list(ai_price_products_df.columns))

    price_candidates = [
        "판매가격",
        "가격",
        "현재가격",
        "price",
        "salePrice",
    ]

    current_price_col = None

    for col in price_candidates:
        if col in ai_price_products_df.columns:
            current_price_col = col
            break

    print()
    print("현재가격 원본 컬럼 :", current_price_col)


    # --------------------------------------------------------------------------
    # 2. 현재 가격 매핑 테이블 생성
    # --------------------------------------------------------------------------

    current_price_map = {}

    if current_price_col is not None:

        for _, r in ai_price_products_df.iterrows():

            vid = str(r.get("vendorItemId", "")).strip()

            if not vid or vid == "nan":
                continue

            value = r.get(current_price_col)

            try:
                value = float(
                    str(value)
                    .replace(",", "")
                    .replace("원", "")
                    .strip()
                )
            except:
                value = None

            current_price_map[vid] = value


    # --------------------------------------------------------------------------
    # 3. 상품별 가격 이력 요약
    # --------------------------------------------------------------------------

    summary_rows = []

    history_df = ai_price_history_df.copy()

    history_df["가격"] = pd.to_numeric(
        history_df["가격"],
        errors="coerce"
    )

    history_df["날짜"] = pd.to_datetime(
        history_df["날짜"],
        errors="coerce"
    )

    history_df = history_df.dropna(
        subset=["가격"]
    )

    history_df = history_df.sort_values(
        ["vendorItemId", "날짜"]
    )


    for vendor_item_id, group in history_df.groupby(
        "vendorItemId",
        sort=False
    ):

        if group.empty:
            continue

        first = group.iloc[0]

        vid = str(vendor_item_id).strip()

        current_price = current_price_map.get(vid)

        min_price = group["가격"].min()
        max_price = group["가격"].max()
        avg_price = group["가격"].mean()

        recent_row = group.iloc[-1]

        recent_price = recent_row["가격"]
        recent_date = recent_row["날짜"]

        row = {
            "검색순위": first.get("검색순위"),
            "검색키워드": first.get("검색키워드"),
            "상품명": first.get("상품명"),
            "상품URL": first.get("상품URL"),

            "productId": first.get("productId"),
            "itemId": first.get("itemId"),
            "vendorItemId": vid,

            "가격이력수": len(group),

            "최저가_180일": min_price,
            "최고가_180일": max_price,
            "평균가_180일": round(avg_price, 0),

            "최근가격_180일": recent_price,
            "최근가격일": recent_date,

            "현재가격": current_price,
        }


        # ----------------------------------------------------------------------
        # 최근가격 대비 현재가격
        # ----------------------------------------------------------------------

        if (
            current_price is not None
            and pd.notna(current_price)
            and recent_price != 0
        ):

            row["최근대비변동률(%)"] = round(
                (
                    current_price - recent_price
                )
                / recent_price
                * 100,
                2
            )

        else:

            row["최근대비변동률(%)"] = None


        # ----------------------------------------------------------------------
        # 최저가 대비 현재가격
        # ----------------------------------------------------------------------

        if (
            current_price is not None
            and pd.notna(current_price)
            and min_price != 0
        ):

            row["최저가대비현재가(%)"] = round(
                (
                    current_price - min_price
                )
                / min_price
                * 100,
                2
            )

        else:

            row["최저가대비현재가(%)"] = None


        # ----------------------------------------------------------------------
        # 최고가 대비 현재가격
        # ----------------------------------------------------------------------

        if (
            current_price is not None
            and pd.notna(current_price)
            and max_price != 0
        ):

            row["최고가대비현재가(%)"] = round(
                (
                    current_price - max_price
                )
                / max_price
                * 100,
                2
            )

        else:

            row["최고가대비현재가(%)"] = None


        summary_rows.append(row)


    # --------------------------------------------------------------------------
    # 4. DataFrame
    # --------------------------------------------------------------------------

    ai_price_summary_df = pd.DataFrame(summary_rows)

    if not ai_price_summary_df.empty:

        ai_price_summary_df = ai_price_summary_df.sort_values(
            "검색순위",
            na_position="last"
        ).reset_index(drop=True)


# ================================================================================
# 5. 결과 확인
# ================================================================================

print()
print("=" * 80)
print("iv-A 완료")
print("=" * 80)

print(
    f"가격 요약 상품 수 : {len(ai_price_summary_df)}"
)

print()

if not ai_price_summary_df.empty:

    display(
        ai_price_summary_df[
            [
                "검색순위",
                "상품명",
                "vendorItemId",
                "가격이력수",
                "최저가_180일",
                "최고가_180일",
                "평균가_180일",
                "최근가격_180일",
                "현재가격",
                "최근대비변동률(%)",
                "최저가대비현재가(%)",
                "최고가대비현재가(%)",
            ]
        ].head(20)
    )

# ================================================================================


In [ ]:
# v. 쿠팡 경쟁사 분석 + AiPrice 가격변화 통합 엑셀 내보내기
# ================================================================================

import pandas as pd
import numpy as np
import re

from datetime import datetime
from google.colab import files

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule


print("=" * 80)
print("⑤ 쿠팡 경쟁사 분석 + AiPrice 통합 엑셀 생성")
print("=" * 80)


# ================================================================================
# 1. 원본 상품 데이터
# ================================================================================

base_df = ai_price_products_df.copy()

print(f"검색결과 원본 : {len(base_df)}개")


# ================================================================================
# 2. 검색키워드 컬럼 확인
# ================================================================================
# 중요
# ------------------------------------------------------------------------------
# 여기서는 검색어를 하나만 뽑지 않습니다.
#
# iv에서 각 HTML 파일별로 이미 입력된
# "검색키워드"를 그대로 사용합니다.
#
# 예:
# 핸드워시
# 핸드워시 250ml
# 핸드워시 500ml
# 핸드워시 2l
# 핸드워시 4l
# ================================================================================

if "검색키워드" not in base_df.columns:

    base_df["검색키워드"] = ""


base_df["검색키워드"] = (
    base_df["검색키워드"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# 빈 검색어 처리
base_df.loc[
    base_df["검색키워드"].eq(""),
    "검색키워드"
] = "검색어없음"


# ================================================================================
# 3. 실제 검색어 목록 확인
# ================================================================================

keyword_series = (
    base_df["검색키워드"]
    .dropna()
    .astype(str)
    .str.strip()
)

keyword_series = keyword_series[
    keyword_series.ne("")
    & keyword_series.ne("검색어없음")
]


unique_search_keywords = (
    keyword_series
    .drop_duplicates()
    .tolist()
)


print()
print("=" * 80)
print("현재 데이터에 들어있는 검색어")
print("=" * 80)

if unique_search_keywords:

    for i, keyword in enumerate(
        unique_search_keywords,
        start=1
    ):

        count = int(
            (
                base_df["검색키워드"]
                == keyword
            ).sum()
        )

        print(
            f"{i}. {keyword} → {count}개"
        )

else:

    print("검색어없음")


# ================================================================================
# 4. 검색요청어
# ================================================================================
# 기존 코드의 문제:
#
# product_df["검색요청어"] = actual_search_keyword
#
# 이렇게 하면 5개 HTML을 합쳐도 전부 첫 번째 검색어가 됩니다.
#
# 수정:
# 검색요청어도 각 행의 검색키워드를 그대로 사용합니다.
# ================================================================================


base_df["검색요청어"] = (
    base_df["검색키워드"]
)


# ================================================================================
# 5. 원본 데이터 확인
# ================================================================================

print()
print("=" * 80)
print("iv → ⑤ 전달 데이터 확인")
print("=" * 80)

preview_columns = [
    "검색순위",
    "검색키워드",
    "검색요청어",
    "상품명",
    "상품URL",
    "productId",
    "itemId",
    "vendorItemId",
]

preview_columns = [
    col
    for col in preview_columns
    if col in base_df.columns
]

if preview_columns:

    print(
        base_df[
            preview_columns
        ]
        .head(30)
        .to_string(index=False)
    )


# ================================================================================
# 6. 검색키워드 통계
# ================================================================================

print()
print("=" * 80)
print("검색키워드별 상품 수")
print("=" * 80)

keyword_counts = (
    base_df["검색키워드"]
    .value_counts(dropna=False)
)

print(keyword_counts)


# ================================================================================
# 7. 검색결과 검수
# ================================================================================

def check_result_relevance(
    search_keyword,
    product_name
):

    keyword = str(
        search_keyword
    ).strip()

    name = str(
        product_name
    ).strip()

    if (
        not keyword
        or keyword == "검색어없음"
    ):

        return "검수불가"

    if not name:

        return "상품명없음"


    # ------------------------------------------------------------
    # 입는생리대 검색 → 물티슈
    # ------------------------------------------------------------

    if (
        "입는생리대" in keyword
        and "물티슈" in name
    ):

        return "검색결과 불일치 가능"


    # ------------------------------------------------------------
    # 생리대 검색 → 명백한 유아용품
    # ------------------------------------------------------------

    if (
        "생리대" in keyword
        and (
            "아기물티슈" in name
            or "유아물티슈" in name
        )
    ):

        return "검색결과 불일치 가능"


    return "정상/확인"


base_df["검색결과검수"] = base_df.apply(

    lambda row:
        check_result_relevance(
            row.get(
                "검색키워드",
                ""
            ),
            row.get(
                "상품명",
                ""
            )
        ),

    axis=1
)


print()
print("검색결과 검수:")

print(
    base_df[
        "검색결과검수"
    ]
    .value_counts(
        dropna=False
    )
)


# ================================================================================
# 8. 분석용 컬럼
# ================================================================================

analysis_columns = [

    "검색순위",
    "검색키워드",
    "검색요청어",
    "검색결과검수",

    "브랜드",
    "상품명",
    "상품URL",

    "상품유형",
    "사이즈",

    "중량(g)",
    "포장당매수",
    "묶음수",
    "총매수",
    "총중량(g)",

    "정가",
    "판매가격",
    "할인율",
    "정가_크롤링상태",
    "1매/개당가격",
    "가격대",

    "리뷰수",
    "리뷰구간",

    "판매수",
    "판매수표시",
    "판매량데이터출처",

    "배송",
    "광고",
    "쿠팡와우",
    "BEST",
    "영역",

    "원문",
]


# ================================================================================
# 9. 기본 상품 데이터 생성
# ================================================================================

product_df = pd.DataFrame(
    index=base_df.index
)


for col in analysis_columns:

    if col in base_df.columns:

        product_df[col] = (
            base_df[col]
        )

    else:

        product_df[col] = ""


# ================================================================================
# 10. 검색 관련 컬럼 확정
# ================================================================================

product_df["검색순위"] = (
    base_df["검색순위"]
    if "검색순위" in base_df.columns
    else ""
)


product_df["검색키워드"] = (
    base_df["검색키워드"]
)


# 중요
# 검색요청어도 행별 검색키워드
product_df["검색요청어"] = (
    base_df["검색키워드"]
)


product_df["검색결과검수"] = (
    base_df["검색결과검수"]
)


# ================================================================================
# 11. 검색어 최종 검증
# ================================================================================

final_request_keywords_check = (
    product_df["검색요청어"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .drop_duplicates()
    .tolist()
)

final_product_keywords_check = (
    product_df["검색키워드"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .drop_duplicates()
    .tolist()
)


print()
print("=" * 80)
print("검색어 최종 검증")
print("=" * 80)

print(
    "검색키워드 :",
    final_product_keywords_check
)

print(
    "검색요청어 :",
    final_request_keywords_check
)


# ================================================================================
# 12. 상품명
# ================================================================================

if "상품명" in base_df.columns:

    product_df["상품명"] = (
        base_df["상품명"]
    )


# ================================================================================
# 13. 상품URL
# ================================================================================

if "상품URL" in base_df.columns:

    product_df["상품URL"] = (
        base_df["상품URL"]
    )


# ================================================================================
# 14. 리뷰수
# ================================================================================

if "리뷰수" in base_df.columns:

    product_df["리뷰수"] = (
        base_df["리뷰수"]
    )


# ================================================================================
# 15. 현재 판매가격
# ================================================================================

if "가격" in base_df.columns:

    product_df["판매가격"] = (
        base_df["가격"]
    )

elif "판매가격" in base_df.columns:

    product_df["판매가격"] = (
        base_df["판매가격"]
    )


# ================================================================================
# 16. 리뷰구간
# ================================================================================

def review_range(x):

    try:

        x = float(x)

        if x < 100:
            return "100개 미만"

        elif x < 1000:
            return "100~999개"

        elif x < 10000:
            return "1천~9,999개"

        elif x < 100000:
            return "1만~9.9만개"

        else:
            return "10만개 이상"

    except:

        return ""


product_df["리뷰구간"] = (
    product_df["리뷰수"]
    .apply(review_range)
)


# ================================================================================
# 17. 판매가격 숫자 변환
# ================================================================================

product_df["판매가격"] = pd.to_numeric(
    product_df["판매가격"],
    errors="coerce"
)


# ================================================================================
# 18. 가격대
# ================================================================================

def price_range(x):

    try:

        x = float(x)

        if x < 5000:
            return "5천원 미만"

        elif x < 10000:
            return "5천~1만원"

        elif x < 15000:
            return "1만~1.5만원"

        elif x < 20000:
            return "1.5만~2만원"

        elif x < 30000:
            return "2만~3만원"

        elif x < 50000:
            return "3만~5만원"

        else:
            return "5만원 이상"

    except:

        return ""


product_df["가격대"] = (
    product_df["판매가격"]
    .apply(price_range)
)


# ================================================================================
# 19. ID 컬럼 안전하게 처리
# ================================================================================

def safe_string_series(
    df,
    column
):

    if column in df.columns:

        return (
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    return pd.Series(
        "",
        index=df.index,
        dtype="object"
    )


product_df["productId"] = (
    safe_string_series(
        base_df,
        "productId"
    )
)


product_df["itemId"] = (
    safe_string_series(
        base_df,
        "itemId"
    )
)


product_df["vendorItemId"] = (
    safe_string_series(
        base_df,
        "vendorItemId"
    )
)


# ================================================================================
# 20. 검색결과 HTML 정가 / 할인율 매칭
# ================================================================================
#
# 기준: vendorItemId
#
# 원칙:
# - 정가: 검색결과 HTML의 basePrice 우선
# - 할인율: 검색결과 HTML의 discountRate
# - 기존 값이 비어 있을 때만 입력
# - 판매가격은 절대 수정하지 않음
# ================================================================================

price_match_count = 0
list_price_fill_count = 0
discount_fill_count = 0
price_unmatched_count = 0
price_no_list_count = 0

if "정가_크롤링상태" not in product_df.columns:
    product_df["정가_크롤링상태"] = ""

original_sale_price_series = (
    product_df["판매가격"].copy()
    if "판매가격" in product_df.columns
    else pd.Series(index=product_df.index, dtype="object")
)

for idx in product_df.index:

    vendor_id = str(
        product_df.at[idx, "vendorItemId"]
    ).strip()

    if (
        not vendor_id
        or vendor_id.lower() in {"nan", "none"}
    ):
        product_df.at[idx, "정가_크롤링상태"] = "vendorItemId 없음"
        continue

    if vendor_id not in all_html_price_data:
        product_df.at[idx, "정가_크롤링상태"] = "vendorItemId 미매칭"
        price_unmatched_count += 1
        continue

    price_match_count += 1

    price_data = all_html_price_data[vendor_id]

    html_list_price = price_data.get("정가")
    html_discount = price_data.get("할인율")

    existing_list_price = product_df.at[idx, "정가"]
    existing_discount = product_df.at[idx, "할인율"]

    list_empty = (
        pd.isna(existing_list_price)
        or str(existing_list_price).strip() == ""
        or str(existing_list_price).strip().lower() in {"nan", "none"}
    )

    discount_empty = (
        pd.isna(existing_discount)
        or str(existing_discount).strip() == ""
        or str(existing_discount).strip().lower() in {"nan", "none"}
    )

    if list_empty and html_list_price is not None:
        product_df.at[idx, "정가"] = html_list_price
        list_price_fill_count += 1

    if discount_empty and html_discount is not None:
        product_df.at[idx, "할인율"] = html_discount
        discount_fill_count += 1

    if html_list_price is not None:
        product_df.at[idx, "정가_크롤링상태"] = "매칭완료"
    else:
        product_df.at[idx, "정가_크롤링상태"] = "vendorItemId 매칭 / 정가 없음"
        price_no_list_count += 1


# 판매가격 보존 검증
if "판매가격" in product_df.columns:
    sale_changed = 0

    for idx in product_df.index:

        old_value = original_sale_price_series.at[idx]
        new_value = product_df.at[idx, "판매가격"]

        def normalize_price_cell(value):
            if pd.isna(value):
                return None

            try:
                numeric = float(str(value).replace(",", "").strip())
                if numeric.is_integer():
                    return int(numeric)
                return numeric
            except Exception:
                return str(value).strip()

        if normalize_price_cell(old_value) != normalize_price_cell(new_value):
            sale_changed += 1

    if sale_changed > 0:
        raise ValueError(
            f"정가/할인율 통합 과정에서 판매가격이 "
            f"{sale_changed}건 변경되었습니다. 저장을 중단합니다."
        )

print()
print("=" * 80)
print("💰 정가 / 할인율 매칭 결과")
print("=" * 80)
print(f"HTML 추출 vendorItemId     : {len(all_html_price_data):,}")
print(f"제품분석 vendorItemId 매칭 : {price_match_count:,}")
print(f"정가 새로 입력              : {list_price_fill_count:,}")
print(f"할인율 새로 입력            : {discount_fill_count:,}")
print(f"정가 없음                   : {price_no_list_count:,}")
print(f"vendorItemId 미매칭         : {price_unmatched_count:,}")
print("판매가격 변경               : 0건")


# ================================================================================
# 21. AiPrice 가격요약 LEFT JOIN
# ================================================================================

price_summary_columns = [

    "가격이력수",
    "최저가_180일",
    "최고가_180일",
    "평균가_180일",
    "최근가격_180일",
    "최근가격일",
    "현재가격",
    "최근대비변동률(%)",
    "최저가대비현재가(%)",
    "최고가대비현재가(%)",

]


if (
    isinstance(
        ai_price_summary_df,
        pd.DataFrame
    )
    and not ai_price_summary_df.empty
    and "vendorItemId"
    in ai_price_summary_df.columns
):

    available_summary_columns = [

        col
        for col in price_summary_columns
        if col in ai_price_summary_df.columns

    ]


    summary_merge = (
        ai_price_summary_df[
            [
                "vendorItemId"
            ]
            + available_summary_columns
        ]
        .copy()
    )


    summary_merge["vendorItemId"] = (
        summary_merge["vendorItemId"]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    summary_merge = (
        summary_merge
        .drop_duplicates(
            subset=[
                "vendorItemId"
            ],
            keep="first"
        )
    )


    product_df = product_df.merge(

        summary_merge,

        on="vendorItemId",

        how="left",

        suffixes=(
            "",
            "_AiPrice"
        )

    )


else:

    for col in price_summary_columns:

        if col not in product_df.columns:

            product_df[col] = np.nan


# ================================================================================
# 21. AiPrice 컬럼 누락 방지
# ================================================================================

for col in price_summary_columns:

    if col not in product_df.columns:

        product_df[col] = np.nan


# ================================================================================
# 22. 가격 차이 지표
# ================================================================================

product_df["현재가-평균가 차이"] = (

    product_df["판매가격"]

    - pd.to_numeric(
        product_df["평균가_180일"],
        errors="coerce"
    )

)


average_price = pd.to_numeric(

    product_df["평균가_180일"],

    errors="coerce"

)


product_df["현재가-평균가 차이율(%)"] = np.where(

    average_price.notna()
    & (average_price != 0),

    (
        product_df["판매가격"]
        - average_price
    )
    / average_price
    * 100,

    np.nan

)


# ================================================================================
# 23. 최종 컬럼 순서
# ================================================================================

final_columns = [

    "검색순위",
    "검색키워드",
    "검색요청어",
    "검색결과검수",

    "브랜드",
    "상품명",
    "상품URL",

    "상품유형",
    "사이즈",

    "중량(g)",
    "포장당매수",
    "묶음수",
    "총매수",
    "총중량(g)",

    "정가",
    "판매가격",
    "할인율",
    "정가_크롤링상태",
    "1매/개당가격",
    "가격대",

    "리뷰수",
    "리뷰구간",

    "판매수",
    "판매수표시",
    "판매량데이터출처",

    "배송",
    "광고",
    "쿠팡와우",
    "BEST",
    "영역",

    "원문",

    "productId",
    "itemId",
    "vendorItemId",

    "가격이력수",
    "최저가_180일",
    "최고가_180일",
    "평균가_180일",
    "최근가격_180일",
    "최근가격일",
    "현재가격",
    "최근대비변동률(%)",
    "최저가대비현재가(%)",
    "최고가대비현재가(%)",

    "현재가-평균가 차이",
    "현재가-평균가 차이율(%)",

]


# 누락 컬럼 방지
for col in final_columns:

    if col not in product_df.columns:

        product_df[col] = ""


final_df = product_df[
    final_columns
].copy()


# ================================================================================
# 24. 가격이력 데이터
# ================================================================================

if (
    isinstance(
        ai_price_history_df,
        pd.DataFrame
    )
):

    history_export_df = (
        ai_price_history_df.copy()
    )

else:

    history_export_df = pd.DataFrame()


history_export_columns = [

    "검색순위",
    "검색키워드",
    "상품명",
    "상품URL",

    "productId",
    "itemId",
    "vendorItemId",

    "날짜",
    "가격",
    "m_price",
    "최저가",

    "adid",
    "price_symbol",

]


if not history_export_df.empty:

    history_export_df = (
        history_export_df[
            [
                col
                for col
                in history_export_columns
                if col
                in history_export_df.columns
            ]
        ]
        .copy()
    )


# ================================================================================
# 25. 가격이력 검색키워드 보완
# ================================================================================
# 중요:
# vendorItemId 하나가 여러 검색어에서 등장할 수 있습니다.
#
# 따라서 가격이력 자체는 AiPrice가 수집한 상품 단위로 유지합니다.
# 검색키워드가 이미 있으면 유지하고,
# 비어 있는 경우에만 제품분석에서 보완합니다.
# ================================================================================

if not history_export_df.empty:

    if "vendorItemId" in history_export_df.columns:

        keyword_map = (
            final_df[
                [
                    "vendorItemId",
                    "검색키워드"
                ]
            ]
            .copy()
        )


        keyword_map["vendorItemId"] = (
            keyword_map["vendorItemId"]
            .fillna("")
            .astype(str)
            .str.strip()
        )


        keyword_map = (
            keyword_map[
                keyword_map["vendorItemId"].ne("")
            ]
            .drop_duplicates(
                subset=[
                    "vendorItemId"
                ]
            )
        )


        if "검색키워드" not in history_export_df.columns:

            history_export_df = (
                history_export_df.merge(
                    keyword_map,
                    on="vendorItemId",
                    how="left"
                )
            )

        else:

            history_export_df["검색키워드"] = (
                history_export_df["검색키워드"]
                .fillna("")
                .astype(str)
                .str.strip()
            )


            history_export_df = (
                history_export_df.merge(
                    keyword_map,
                    on="vendorItemId",
                    how="left",
                    suffixes=(
                        "",
                        "_제품분석"
                    )
                )
            )


            empty_history_keyword = (
                history_export_df[
                    "검색키워드"
                ]
                .eq("")
            )


            history_export_df.loc[
                empty_history_keyword,
                "검색키워드"
            ] = history_export_df.loc[
                empty_history_keyword,
                "검색키워드_제품분석"
            ]


            if (
                "검색키워드_제품분석"
                in history_export_df.columns
            ):

                history_export_df = (
                    history_export_df
                    .drop(
                        columns=[
                            "검색키워드_제품분석"
                        ]
                    )
                )


# ================================================================================
# 26. 가격요약 데이터
# ================================================================================

if isinstance(
    ai_price_summary_df,
    pd.DataFrame
):

    summary_export_df = (
        ai_price_summary_df.copy()
    )

else:

    summary_export_df = pd.DataFrame()


# ================================================================================
# 27. 시장요약
# ================================================================================
# 여러 검색어를 모두 표시
# ================================================================================

if len(product_df):

    market_keywords = (
        product_df["검색키워드"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    market_keywords = (
        market_keywords[
            market_keywords.ne("")
            & market_keywords.ne("검색어없음")
        ]
        .drop_duplicates()
        .tolist()
    )

else:

    market_keywords = []


if market_keywords:

    market_keyword = (
        " / ".join(
            market_keywords
        )
    )

else:

    market_keyword = "검색어없음"


# 실제 검색 요청어도 동일하게 표시
actual_search_keywords = (
    market_keywords.copy()
)


actual_search_keyword_text = (
    " / ".join(
        actual_search_keywords
    )
    if actual_search_keywords
    else "검색어없음"
)


# ================================================================================
# 28. 현재 가격
# ================================================================================

current_prices = pd.to_numeric(
    product_df["판매가격"],
    errors="coerce"
)


# ================================================================================
# 29. AiPrice 조회 상품 수
# ================================================================================

if "vendorItemId" in base_df.columns:

    ai_price_product_count = (
        base_df["vendorItemId"]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace(
            "",
            np.nan
        )
        .nunique()
    )

else:

    ai_price_product_count = 0


# ================================================================================
# 30. 가격이력 상품 수
# ================================================================================

if (
    isinstance(
        ai_price_summary_df,
        pd.DataFrame
    )
    and not ai_price_summary_df.empty
    and "vendorItemId"
    in ai_price_summary_df.columns
):

    history_product_count = (
        ai_price_summary_df[
            "vendorItemId"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace(
            "",
            np.nan
        )
        .nunique()
    )

else:

    history_product_count = 0


# ================================================================================
# 31. 180일 평균
# ================================================================================

if (
    isinstance(
        ai_price_summary_df,
        pd.DataFrame
    )
    and "평균가_180일"
    in ai_price_summary_df.columns
):

    avg_180_price = pd.to_numeric(
        ai_price_summary_df[
            "평균가_180일"
        ],
        errors="coerce"
    ).mean()

else:

    avg_180_price = np.nan


# ================================================================================
# 32. 180일 최저가 평균
# ================================================================================

if (
    isinstance(
        ai_price_summary_df,
        pd.DataFrame
    )
    and "최저가_180일"
    in ai_price_summary_df.columns
):

    min_180_price = pd.to_numeric(
        ai_price_summary_df[
            "최저가_180일"
        ],
        errors="coerce"
    ).mean()

else:

    min_180_price = np.nan


# ================================================================================
# 33. 180일 최고가 평균
# ================================================================================

if (
    isinstance(
        ai_price_summary_df,
        pd.DataFrame
    )
    and "최고가_180일"
    in ai_price_summary_df.columns
):

    max_180_price = pd.to_numeric(
        ai_price_summary_df[
            "최고가_180일"
        ],
        errors="coerce"
    ).mean()

else:

    max_180_price = np.nan


# ================================================================================
# 34. 전체 가격이력 수
# ================================================================================

history_count = (
    len(ai_price_history_df)
    if isinstance(
        ai_price_history_df,
        pd.DataFrame
    )
    else 0
)


# ================================================================================
# 35. 검색어별 상품 수 요약
# ================================================================================

keyword_summary_text = []

for keyword in market_keywords:

    count = int(
        (
            product_df["검색키워드"]
            == keyword
        ).sum()
    )

    keyword_summary_text.append(
        f"{keyword}: {count}개"
    )


keyword_summary_text = (
    " / ".join(
        keyword_summary_text
    )
    if keyword_summary_text
    else "없음"
)


# ================================================================================
# 36. 시장요약
# ================================================================================

market_summary = pd.DataFrame({

    "지표": [

        "검색 키워드",

        "실제 검색 요청어",

        "검색 키워드별 상품 수",

        "검색결과 상품 수",

        "검색결과 불일치 가능 상품 수",

        "AiPrice 조회 상품 수",

        "AiPrice 가격이력 확보 상품 수",

        "전체 가격이력 수",

        "현재가격 평균",
        "현재가격 최저",
        "현재가격 최고",

        "180일 평균가격 평균",
        "180일 최저가격 평균",
        "180일 최고가격 평균",

    ],

    "값": [

        market_keyword,

        actual_search_keyword_text,

        keyword_summary_text,

        len(base_df),

        int(
            (
                product_df[
                    "검색결과검수"
                ]
                == "검색결과 불일치 가능"
            )
            .sum()
        ),

        ai_price_product_count,

        history_product_count,

        history_count,

        current_prices.mean(),

        current_prices.min(),

        current_prices.max(),

        avg_180_price,

        min_180_price,

        max_180_price,

    ]

})


# ================================================================================
# 37. 데이터 품질
# ================================================================================

quality_rows = []


for col in final_columns:

    if col not in final_df.columns:

        non_empty = 0

    else:

        series = final_df[col]

        if pd.api.types.is_numeric_dtype(
            series
        ):

            non_empty = (
                series.notna().sum()
            )

        else:

            non_empty = (
                series
                .fillna("")
                .astype(str)
                .str.strip()
                .replace(
                    [
                        "nan",
                        "None"
                    ],
                    ""
                )
                .ne("")
                .sum()
            )


    quality_rows.append({

        "컬럼": col,

        "전체건수": len(final_df),

        "값있음": non_empty,

        "빈칸": (
            len(final_df)
            - non_empty
        ),

        "완성률(%)": (

            round(
                non_empty
                / len(final_df)
                * 100,
                1
            )

            if len(final_df)

            else 0

        )

    })


quality_df = pd.DataFrame(
    quality_rows
)


# ================================================================================
# 38. 파일명용 검색키워드
# ================================================================================

if market_keywords:

    if len(market_keywords) == 1:

        filename_keyword = (
            market_keywords[0]
        )

    else:

        # 여러 검색어를 처리한 경우
        # 첫 번째 대표 검색어 + 검색어 개수
        filename_keyword = (
            f"{market_keywords[0]}_외"
            f"{len(market_keywords) - 1}개"
        )

else:

    filename_keyword = "검색어없음"


safe_keyword = re.sub(
    r'[\\/:*?"<>|]',
    "_",
    filename_keyword
)


timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M"
)


filename = (
    f"쿠팡_{safe_keyword}_"
    f"경쟁사분석_AiPrice_"
    f"{timestamp}.xlsx"
)


# ================================================================================
# 39. 엑셀 저장
# ================================================================================

with pd.ExcelWriter(
    filename,
    engine="openpyxl"
) as writer:

    final_df.to_excel(
        writer,
        sheet_name="제품분석",
        index=False
    )

    history_export_df.to_excel(
        writer,
        sheet_name="가격이력",
        index=False
    )

    summary_export_df.to_excel(
        writer,
        sheet_name="가격요약",
        index=False
    )

    market_summary.to_excel(
        writer,
        sheet_name="시장요약",
        index=False
    )

    quality_df.to_excel(
        writer,
        sheet_name="데이터품질",
        index=False
    )


# ================================================================================
# 40. 엑셀 스타일링
# ================================================================================

wb = load_workbook(
    filename
)


header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)


header_font = Font(
    color="FFFFFF",
    bold=True
)


thin_border = Border(
    bottom=Side(
        style="thin",
        color="D9E1F2"
    )
)


# ================================================================================
# 41. 모든 시트 기본 스타일
# ================================================================================

for ws in wb.worksheets:

    # 헤더
    for cell in ws[1]:

        cell.fill = header_fill

        cell.font = header_font

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


    # 필터
    if (
        ws.max_row >= 1
        and ws.max_column >= 1
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # 첫 행 고정
    ws.freeze_panes = "A2"


    # 헤더 높이
    ws.row_dimensions[1].height = 30


    # 셀 기본 스타일
    for row in ws.iter_rows():

        for cell in row:

            cell.border = thin_border

            cell.alignment = Alignment(
                vertical="center"
            )


    # 열 너비
    for col_cells in ws.columns:

        max_length = 0

        for cell in col_cells:

            try:

                value_length = len(
                    str(cell.value)
                )

                if (
                    value_length
                    > max_length
                ):

                    max_length = (
                        value_length
                    )

            except:

                pass


        width = min(

            max(
                max_length + 2,
                10
            ),

            50

        )


        ws.column_dimensions[
            get_column_letter(
                col_cells[0].column
            )
        ].width = width


# ================================================================================
# 42. 제품분석 숫자 포맷
# ================================================================================

ws = wb["제품분석"]


header_map = {

    cell.value: cell.column

    for cell in ws[1]

}


currency_columns = [

    "정가",
    "판매가격",
    "1매/개당가격",

    "최저가_180일",
    "최고가_180일",
    "평균가_180일",
    "최근가격_180일",
    "현재가격",

    "현재가-평균가 차이",

]


percent_columns = [

    "할인율",

    "최근대비변동률(%)",
    "최저가대비현재가(%)",
    "최고가대비현재가(%)",

    "현재가-평균가 차이율(%)",

]


for col in currency_columns:

    if col in header_map:

        column_index = (
            header_map[col]
        )

        for row in range(
            2,
            ws.max_row + 1
        ):

            ws.cell(
                row=row,
                column=column_index
            ).number_format = (
                "#,##0"
            )


for col in percent_columns:

    if col in header_map:

        column_index = (
            header_map[col]
        )

        for row in range(
            2,
            ws.max_row + 1
        ):

            ws.cell(
                row=row,
                column=column_index
            ).number_format = (
                "0.00"
            )


# ================================================================================
# 43. 검색결과 불일치 가능 행 강조
# ================================================================================

if "검색결과검수" in header_map:

    col_idx = header_map[
        "검색결과검수"
    ]


    warning_fill = PatternFill(
        fill_type="solid",
        fgColor="FFF2CC"
    )


    for row in range(
        2,
        ws.max_row + 1
    ):

        cell = ws.cell(
            row=row,
            column=col_idx
        )


        if (
            cell.value
            == "검색결과 불일치 가능"
        ):

            for col in range(
                1,
                ws.max_column + 1
            ):

                ws.cell(
                    row=row,
                    column=col
                ).fill = warning_fill


# ================================================================================
# 44. 가격변동률 조건부 서식
# ================================================================================

if "최근대비변동률(%)" in header_map:

    col_idx = header_map[
        "최근대비변동률(%)"
    ]

    col_letter = (
        get_column_letter(
            col_idx
        )
    )


    if ws.max_row >= 2:

        ws.conditional_formatting.add(

            f"{col_letter}2:"
            f"{col_letter}{ws.max_row}",

            ColorScaleRule(

                start_type="min",
                start_color="63BE7B",

                mid_type="num",
                mid_value=0,
                mid_color="FFEB84",

                end_type="max",
                end_color="F8696A"

            )

        )


# ================================================================================
# 45. 가격이력 날짜 / 가격 포맷
# ================================================================================

ws_history = wb["가격이력"]


if ws_history.max_row >= 1:

    history_map = {

        cell.value: cell.column

        for cell in ws_history[1]

    }


    if "날짜" in history_map:

        col_idx = history_map[
            "날짜"
        ]

        for row in range(
            2,
            ws_history.max_row + 1
        ):

            ws_history.cell(
                row=row,
                column=col_idx
            ).number_format = (
                "yyyy-mm-dd hh:mm"
            )


    if "가격" in history_map:

        col_idx = history_map[
            "가격"
        ]

        for row in range(
            2,
            ws_history.max_row + 1
        ):

            ws_history.cell(
                row=row,
                column=col_idx
            ).number_format = (
                "#,##0"
            )


# ================================================================================
# 46. 가격요약 포맷
# ================================================================================

ws_summary = wb["가격요약"]


if ws_summary.max_row >= 1:

    summary_map = {

        cell.value: cell.column

        for cell in ws_summary[1]

    }


    for col in [

        "최저가_180일",
        "최고가_180일",
        "평균가_180일",
        "최근가격_180일",
        "현재가격",

    ]:

        if col in summary_map:

            col_idx = summary_map[
                col
            ]


            for row in range(
                2,
                ws_summary.max_row + 1
            ):

                ws_summary.cell(
                    row=row,
                    column=col_idx
                ).number_format = (
                    "#,##0"
                )


    for col in [

        "최근대비변동률(%)",
        "최저가대비현재가(%)",
        "최고가대비현재가(%)",

    ]:

        if col in summary_map:

            col_idx = summary_map[
                col
            ]


            for row in range(
                2,
                ws_summary.max_row + 1
            ):

                ws_summary.cell(
                    row=row,
                    column=col_idx
                ).number_format = (
                    "0.00"
                )


# ================================================================================
# 47. 시장요약 숫자 포맷
# ================================================================================

ws_market = wb[
    "시장요약"
]


for row in range(
    2,
    ws_market.max_row + 1
):

    metric = ws_market.cell(
        row=row,
        column=1
    ).value


    if metric and any(

        keyword in str(metric)

        for keyword in [

            "가격",
            "상품 수",
            "상품수",
            "가격이력 수",
            "불일치 가능 상품 수",

        ]

    ):

        # 문자열인 검색 키워드/검색어별 상품 수는 제외
        if metric not in [
            "검색 키워드",
            "실제 검색 요청어",
            "검색 키워드별 상품 수",
        ]:

            ws_market.cell(
                row=row,
                column=2
            ).number_format = (
                "#,##0"
            )


# ================================================================================
# 48. 저장
# ================================================================================

wb.save(
    filename
)


# ================================================================================
# 49. 최종 결과 출력
# ================================================================================

print()
print("=" * 80)
print("⑤ 엑셀 생성 완료")
print("=" * 80)

print(
    f"파일명 : {filename}"
)

print()

print("시트:")

print(
    "  1. 제품분석       → "
    f"{len(final_df)}개 검색결과 + AiPrice 가격지표"
)

print(
    "  2. 가격이력       → "
    f"{len(history_export_df):,}건 가격 변화"
)

print(
    "  3. 가격요약       → "
    f"{len(summary_export_df):,}개 상품 가격 요약"
)

print(
    "  4. 시장요약       → "
    "전체 검색어 시장 가격 통계"
)

print(
    "  5. 데이터품질     → "
    "컬럼별 데이터 확보율"
)


# ================================================================================
# 50. 최종 검색어 확인
# ================================================================================

print()
print("=" * 80)
print("최종 검색어 확인")
print("=" * 80)

print(
    "실제 쿠팡 검색어 :"
)

for keyword in final_product_keywords_check:

    count = int(
        (
            final_df["검색키워드"]
            == keyword
        ).sum()
    )

    print(
        f"  - {keyword} → {count}개"
    )


print()
print(
    "제품분석 검색키워드 :"
)

print(
    final_product_keywords_check
)


print()
print(
    "제품분석 검색요청어 :"
)

print(
    final_request_keywords_check
)


# ================================================================================
# 51. 최종 상품 샘플
# ================================================================================

print()
print("=" * 80)
print("최종 제품분석 확인")
print("=" * 80)

sample_columns = [

    "검색순위",
    "검색키워드",
    "검색요청어",
    "검색결과검수",
    "상품명",

]

sample_columns = [

    col
    for col in sample_columns
    if col in final_df.columns

]


print(

    final_df[
        sample_columns
    ]
    .head(20)
    .to_string(
        index=False
    )

)


# ================================================================================
# 52. 검색어별 실제 결과 확인
# ================================================================================

print()
print("=" * 80)
print("검색어별 실제 결과")
print("=" * 80)


for keyword in final_product_keywords_check:

    temp = final_df[
        final_df["검색키워드"]
        == keyword
    ]

    print()
    print(
        f"[{keyword}] "
        f"{len(temp)}개"
    )

    if len(temp):

        print(

            temp[
                [
                    "검색순위",
                    "상품명"
                ]
            ]
            .head(5)
            .to_string(
                index=False
            )

        )


# ================================================================================
# 53. 중간 파일 다운로드 생략
# ================================================================================
#
# 이후 상세페이지 수집과 상품명 기반 빈칸 보완을 같은 실행에서 진행한 뒤
# 최종 결과 파일만 마지막에 다운로드한다.
#



In [ ]:
# ================================================================
# [LAST CELL] 상품목록 최종 정리 / 저장
# ================================================================

# ================================================================
# 0. 기본 환경
# ================================================================

import os
import re
import shutil
import pandas as pd

from google.colab import files

print("=" * 70)
print("📂 상품목록 최종 정리 시작")
print("=" * 70)


# ================================================================
# 1. Google Drive 연결
# ================================================================

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"⚠️ Google Drive 연결 확인 중: {e}")


# ================================================================
# 2. 저장 경로
# ================================================================

DRIVE_PRODUCT_DIR = (
    "/content/drive/MyDrive/YOUR_PROJECT_FOLDER/data_coupang"
)

LOCAL_PRODUCT_DIR = "/content/coupang_product_output"

os.makedirs(LOCAL_PRODUCT_DIR, exist_ok=True)
os.makedirs(DRIVE_PRODUCT_DIR, exist_ok=True)

print(f"☁️ Google Drive 저장 경로")
print(f"   {DRIVE_PRODUCT_DIR}")
print()


# ================================================================
# 3. 입력 Excel 파일 확인
# ================================================================

# 기존 셀에서 input_file이 만들어져 있으면 그대로 사용
if "input_file" in globals() and input_file:
    if os.path.exists(input_file):
        print("✅ 기존 input_file 사용")
        print(f"   {input_file}")
    else:
        print("⚠️ 기존 input_file 경로가 존재하지 않습니다.")
        input_file = None
else:
    input_file = None


# ------------------------------------------------
# input_file이 없으면 현재 작업 폴더에서 Excel 검색
# ------------------------------------------------

if not input_file:

    candidate_dirs = [
        "/content",
        "/content/coupang_product_output",
    ]

    excel_candidates = []

    for directory in candidate_dirs:

        if not os.path.exists(directory):
            continue

        for filename in os.listdir(directory):

            if not filename.lower().endswith((".xlsx", ".xls")):
                continue

            # 결과 파일은 입력 후보에서 제외
            if "상품명규칙_빈칸보완_중복제거" in filename:
                continue

            full_path = os.path.join(directory, filename)

            if os.path.isfile(full_path):
                excel_candidates.append(full_path)


    # ------------------------------------------------
    # Excel이 하나만 있으면 자동 선택
    # ------------------------------------------------

    if len(excel_candidates) == 1:

        input_file = excel_candidates[0]

        print("✅ Excel 파일 자동 발견")
        print(f"   {input_file}")

    # ------------------------------------------------
    # 여러 개면 사용자 선택
    # ------------------------------------------------

    elif len(excel_candidates) > 1:

        print("📑 사용 가능한 Excel 파일")
        print()

        for i, path in enumerate(excel_candidates, 1):
            print(f"   [{i}] {os.path.basename(path)}")

        print()

        # 제품분석 시트가 있는 파일을 우선 탐색
        selected = None

        for path in excel_candidates:

            try:
                xls_check = pd.ExcelFile(path)

                if "제품분석" in xls_check.sheet_names:
                    selected = path
                    break

            except Exception:
                pass

        if selected:

            input_file = selected

            print("🎯 '제품분석' 시트가 있는 파일을 자동 선택했습니다.")
            print(f"   {input_file}")

        else:

            print("📂 Excel 파일이 여러 개 발견되었습니다.")
            print("   가장 최근 수정된 파일을 사용합니다.")

            input_file = max(
                excel_candidates,
                key=os.path.getmtime
            )

            print(f"   {input_file}")

    # ------------------------------------------------
    # 아무것도 없으면 업로드
    # ------------------------------------------------

    else:

        print("📂 처리할 Excel 파일을 찾지 못했습니다.")
        print()
        print("Excel 파일을 업로드해주세요.")
        print()

        uploaded = files.upload()

        excel_uploaded = [
            name
            for name in uploaded.keys()
            if name.lower().endswith((".xlsx", ".xls"))
        ]

        if not excel_uploaded:
            raise FileNotFoundError(
                "❌ Excel 파일이 업로드되지 않았습니다."
            )

        input_file = os.path.abspath(excel_uploaded[0])

        print()
        print("✅ 업로드된 Excel 사용")
        print(f"   {input_file}")


# ================================================================
# 4. 입력 파일 최종 확인
# ================================================================

if not input_file:
    raise ValueError(
        "❌ input_file을 확인할 수 없습니다."
    )

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"❌ 입력 Excel 파일을 찾을 수 없습니다.\n"
        f"경로: {input_file}"
    )

print()
print("=" * 70)
print("📂 입력 파일 확인 완료")
print("=" * 70)
print(f"입력 파일 : {input_file}")
print()


# ================================================================
# 5. Excel 읽기
# ================================================================

xls = pd.ExcelFile(input_file)

print("📑 Excel 시트")

for sheet in xls.sheet_names:
    print(f"   - {sheet}")

print()


if "제품분석" in xls.sheet_names:

    df = pd.read_excel(
        input_file,
        sheet_name="제품분석"
    )

    target_sheet = "제품분석"

else:

    df = pd.read_excel(
        input_file,
        sheet_name=xls.sheet_names[0]
    )

    target_sheet = xls.sheet_names[0]


print(f"🎯 처리 시트 : {target_sheet}")
print(f"📊 원본 행 수 : {len(df):,}")
print()


# ================================================================
# 6. 문자열 정리 함수
# ================================================================

def clean_value(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() in ["nan", "none", "null"]:
        return ""

    return x


def is_blank(x):
    return clean_value(x) == ""


def to_number(x):

    try:

        text = clean_value(x)
        text = text.replace(",", "")

        if text == "":
            return None

        return float(text)

    except Exception:
        return None


# ================================================================
# 7. 기존 불필요 컬럼 제거
# ================================================================

remove_cols = [
    "배송",
    "광고",
    "쿠팡와우",
    "BEST",
    "영역",
    "원문",
    "productId",
]

existing_remove_cols = [
    c for c in remove_cols
    if c in df.columns
]

if existing_remove_cols:

    print("🗑️ 제거 컬럼")

    for c in existing_remove_cols:
        print(f"   - {c}")

    df = df.drop(
        columns=existing_remove_cols
    )

else:

    print("🗑️ 제거할 불필요 컬럼 없음")

print()


# ================================================================
# 8. 필요한 컬럼 생성
# ================================================================

required_columns = [
    "브랜드",
    "상품명",
    "상품유형",
    "사이즈",
    "중량(g)",
    "포장당매수",
    "묶음수",
    "총매수",
    "1매/개당가격",
]

for col in required_columns:

    if col not in df.columns:
        df[col] = ""


# ================================================================
# 9. 상품명 기반 빈칸 보완 함수
# ================================================================

def extract_brand(product_name):

    text = clean_value(product_name)

    if not text:
        return ""

    patterns = [
        r"^([가-힣A-Za-z0-9&.\-]+)\s+",
    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            text
        )

        if m:

            candidate = m.group(1).strip()

            if len(candidate) >= 2:
                return candidate

    return ""


def extract_weight(product_name):

    text = clean_value(product_name)

    if not text:
        return ""

    # kg
    m = re.search(
        r"(?<!\d)(\d+(?:\.\d+)?)\s*kg\b",
        text,
        flags=re.I
    )

    if m:

        try:

            value = float(m.group(1)) * 1000

            if value.is_integer():
                return str(int(value))

            return str(value)

        except Exception:
            pass

    # g
    m = re.search(
        r"(?<!\d)(\d+(?:\.\d+)?)\s*g\b",
        text,
        flags=re.I
    )

    if m:
        return m.group(1)

    return ""


def extract_size(product_name):

    text = clean_value(product_name)

    if not text:
        return ""

    patterns = [

        r"(\d+(?:\.\d+)?\s*[xX×]\s*\d+(?:\.\d+)?\s*(?:cm|mm))",

        r"(\d+(?:\.\d+)?\s*(?:cm|mm))",

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            flags=re.I
        )

        if m:
            return m.group(1).strip()

    return ""


def extract_pack_count(product_name):

    text = clean_value(product_name)

    if not text:
        return ""

    patterns = [

        r"(\d+)\s*개입",

        r"(\d+)\s*매입",

        r"(\d+)\s*개\s*(?:입|포장)",

        r"(\d+)\s*매\s*(?:입|포장)",

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            flags=re.I
        )

        if m:
            return m.group(1)

    return ""


def extract_bundle_count(product_name):

    text = clean_value(product_name)

    if not text:
        return ""

    # 중요:
    # 3종 / 2종은 묶음수가 아니라 구성 종류이므로 제외
    patterns = [

        r"(\d+)\s*묶음",

        r"(\d+)\s*팩",

        r"(\d+)\s*세트",

        r"(\d+)\s*박스",

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            flags=re.I
        )

        if m:
            return m.group(1)

    return ""


# ================================================================
# 10. 빈칸 보완
# ================================================================

print("🔎 상품명 기반 빈칸 보완 시작")
print()

fill_counts = {
    "브랜드": 0,
    "중량(g)": 0,
    "사이즈": 0,
    "포장당매수": 0,
    "묶음수": 0,
}


for idx in df.index:

    product_name = clean_value(
        df.at[idx, "상품명"]
    )

    if not product_name:
        continue


    # ------------------------------------------------
    # 브랜드
    # ------------------------------------------------

    if is_blank(df.at[idx, "브랜드"]):

        value = extract_brand(
            product_name
        )

        if value:

            df.at[idx, "브랜드"] = value
            fill_counts["브랜드"] += 1


    # ------------------------------------------------
    # 중량
    # ------------------------------------------------

    if is_blank(df.at[idx, "중량(g)"]):

        value = extract_weight(
            product_name
        )

        if value:

            df.at[idx, "중량(g)"] = value
            fill_counts["중량(g)"] += 1


    # ------------------------------------------------
    # 사이즈
    # ------------------------------------------------

    if is_blank(df.at[idx, "사이즈"]):

        value = extract_size(
            product_name
        )

        if value:

            df.at[idx, "사이즈"] = value
            fill_counts["사이즈"] += 1


    # ------------------------------------------------
    # 포장당 매수
    # ------------------------------------------------

    if is_blank(df.at[idx, "포장당매수"]):

        value = extract_pack_count(
            product_name
        )

        if value:

            df.at[idx, "포장당매수"] = value
            fill_counts["포장당매수"] += 1


    # ------------------------------------------------
    # 묶음수
    # ------------------------------------------------

    if is_blank(df.at[idx, "묶음수"]):

        value = extract_bundle_count(
            product_name
        )

        if value:

            df.at[idx, "묶음수"] = value
            fill_counts["묶음수"] += 1


print("✅ 빈칸 보완 결과")

for col, count in fill_counts.items():

    print(
        f"   {col:<12} : {count:,}건"
    )

print()


# ================================================================
# 11. 총매수 계산
# ================================================================

if "총매수" in df.columns:

    calculated_total = 0

    for idx in df.index:

        pack = to_number(
            df.at[idx, "포장당매수"]
        )

        bundle = to_number(
            df.at[idx, "묶음수"]
        )

        if (
            pack is not None
            and bundle is not None
        ):

            total = pack * bundle

            if total.is_integer():
                total = int(total)

            df.at[idx, "총매수"] = total

            calculated_total += 1

print(
    f"✅ 총매수 계산 완료 : {calculated_total:,}건"
)
print()


# ================================================================
# 12. 1매/개당가격 계산
# ================================================================

calculated_unit_price = 0

if (
    "판매가격" in df.columns
    and "총매수" in df.columns
    and "1매/개당가격" in df.columns
):

    for idx in df.index:

        price = to_number(
            df.at[idx, "판매가격"]
        )

        total_count = to_number(
            df.at[idx, "총매수"]
        )

        if (
            price is not None
            and total_count is not None
            and total_count > 0
        ):

            unit_price = price / total_count

            df.at[
                idx,
                "1매/개당가격"
            ] = round(
                unit_price,
                2
            )

            calculated_unit_price += 1


print(
    f"✅ 1매/개당가격 계산 완료 : "
    f"{calculated_unit_price:,}건"
)
print()


# ================================================================
# 13. 중복 제거
#
# 조건:
# vendorItemId + 포장당매수 + 묶음수
#
# 세 값이 모두 존재하는 행만 중복 판정
# ================================================================

before_dedup = len(df)

dedup_cols = [
    "vendorItemId",
    "포장당매수",
    "묶음수",
]

if all(
    col in df.columns
    for col in dedup_cols
):

    valid_mask = (

        df["vendorItemId"].apply(
            lambda x: not is_blank(x)
        )

        &

        df["포장당매수"].apply(
            lambda x: not is_blank(x)
        )

        &

        df["묶음수"].apply(
            lambda x: not is_blank(x)
        )

    )

    valid_df = df[
        valid_mask
    ].copy()

    invalid_df = df[
        ~valid_mask
    ].copy()


    # 세 조건이 모두 존재하는 행만 중복 제거
    valid_df = valid_df.drop_duplicates(
        subset=dedup_cols,
        keep="first"
    )


    df = pd.concat(
        [
            valid_df,
            invalid_df
        ],
        ignore_index=True
    )

else:

    print(
        "⚠️ 중복 제거 기준 컬럼이 부족하여 "
        "중복 제거를 건너뜁니다."
    )


after_dedup = len(df)

print("🧹 중복 제거")
print(f"   제거 전 : {before_dedup:,}행")
print(f"   제거 후 : {after_dedup:,}행")
print(
    f"   제거 수 : "
    f"{before_dedup - after_dedup:,}행"
)
print()


# ================================================================
# 14. 컬럼 순서 정리
# ================================================================

preferred_order = [

    "검색키워드",
    "검색순위",
    "vendorItemId",

    "브랜드",
    "상품명",
    "상품유형",

    "사이즈",
    "중량(g)",
    "포장당매수",
    "묶음수",
    "총매수",

    "정가",
    "할인율",
    "판매가격",
    "1매/개당가격",

    "리뷰수",
    "판매수",

    "상품URL",
    "검색요청어",
    "검색결과검수",

    "총중량(g)",
    "정가_크롤링상태",

    "가격대",
    "리뷰구간",
    "판매수표시",
    "판매량데이터출처",

    "itemId",

    "가격이력수",
    "최저가_180일",
    "최고가_180일",
    "평균가_180일",
    "최근가격_180일",
    "최근가격일",
    "현재가격",

    "최근대비변동률(%)",
    "최저가대비현재가(%)",
    "최고가대비현재가(%)",

    "현재가-평균가 차이",
    "현재가-평균가 차이율(%)",

]


existing_preferred = [
    col
    for col in preferred_order
    if col in df.columns
]


remaining_cols = [
    col
    for col in df.columns
    if col not in existing_preferred
]


df = df[
    existing_preferred + remaining_cols
]


print("📋 컬럼 순서 정리 완료")
print(f"   컬럼 수 : {len(df.columns)}")
print()


# ================================================================
# 15. 원본 Excel의 다른 시트 보존
# ================================================================

print("📑 기존 Excel 시트 보존 중...")

all_sheets = {}

for sheet_name in xls.sheet_names:

    if sheet_name == target_sheet:
        continue

    try:

        all_sheets[sheet_name] = pd.read_excel(
            input_file,
            sheet_name=sheet_name
        )

    except Exception as e:

        print(
            f"⚠️ 시트 읽기 실패: "
            f"{sheet_name} / {e}"
        )


print(
    f"   보존 시트 : "
    f"{len(all_sheets)}개"
)
print()


# ================================================================
# 16. 파일명 생성
# ================================================================

base_name = os.path.splitext(
    os.path.basename(input_file)
)[0]


output_filename = (
    f"{base_name}_상품명규칙_빈칸보완_중복제거.xlsx"
)


# ------------------------------------------------
# 로컬 저장
# ------------------------------------------------

output_file = os.path.join(
    LOCAL_PRODUCT_DIR,
    output_filename
)


# ------------------------------------------------
# Google Drive 저장
# ------------------------------------------------

drive_output_file = os.path.join(
    DRIVE_PRODUCT_DIR,
    output_filename
)


# ================================================================
# 17. Excel 저장
# ================================================================

print("=" * 70)
print("💾 Excel 저장")
print("=" * 70)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # ------------------------------------------------
    # 제품분석
    # ------------------------------------------------

    df.to_excel(
        writer,
        sheet_name=target_sheet,
        index=False
    )


    # ------------------------------------------------
    # 나머지 시트
    # ------------------------------------------------

    for sheet_name, sheet_df in all_sheets.items():

        safe_sheet_name = str(
            sheet_name
        )[:31]

        sheet_df.to_excel(
            writer,
            sheet_name=safe_sheet_name,
            index=False
        )


print("✅ 로컬 저장 완료")
print(f"   {output_file}")
print()


# ================================================================
# 18. Google Drive 저장
# ================================================================

os.makedirs(
    DRIVE_PRODUCT_DIR,
    exist_ok=True
)


shutil.copy2(
    output_file,
    drive_output_file
)


print("☁️ Google Drive 저장 완료")
print(f"   {drive_output_file}")
print()


# ================================================================
# 19. 저장 파일 검증
# ================================================================

if not os.path.exists(output_file):

    raise FileNotFoundError(
        "❌ 로컬 결과 파일이 생성되지 않았습니다."
    )


if not os.path.exists(drive_output_file):

    raise FileNotFoundError(
        "❌ Google Drive 결과 파일이 생성되지 않았습니다."
    )


local_size = os.path.getsize(
    output_file
)

drive_size = os.path.getsize(
    drive_output_file
)


print(
    f"✅ 로컬 파일 확인 : "
    f"{local_size:,} bytes"
)

print(
    f"✅ Drive 파일 확인 : "
    f"{drive_size:,} bytes"
)

print()


# ================================================================
# 20. 최종 결과 요약
# ================================================================

print("=" * 70)
print("🎉 상품목록 처리 완료")
print("=" * 70)

print(
    f"📊 최종 상품 수 : "
    f"{len(df):,}개"
)

print(
    f"📋 최종 컬럼 수 : "
    f"{len(df.columns):,}개"
)

print()

print("💾 저장 위치")

print("1. 로컬")
print(f"   {output_file}")

print()

print("2. Google Drive")
print(f"   {drive_output_file}")

print()

print("=" * 70)


# ================================================================
# 21. 로컬 결과 자동 다운로드
# ================================================================

files.download(
    output_file
)